# 第 76 课：推理 SLO、尾延迟、准入控制与多租户公平性

承接第75课的真实前缀缓存＋物理 KV 页池。这一课在同一套离线 CPU 因果 Transformer 上加入**有界请求队列、首 token 预算、端到端完成预算、租户轮转和提前拒绝决策**。

**本课要解决的问题：** P95/P99 是从哪些请求算出来的？排队失败是否从分母中消失？FIFO、轮转、EDF 和 SLO 提前拒绝有何区别？缓存和物理容量的账本能否始终守恒？

本讲义包含完整源码与实际训练以外的推理调度实验，**不依赖第75课文件、不下载预训练模型或用户数据**。模型 logits 使用真实 CPU 前向与物理 KV，调度计时却是人为定义的**逻辑 tick**，不是 GPU 毫秒、网络延迟或生产性能成绩。

## 0. 阅读顺序与运行方法

从新 Python 内核依次运行。只需已经配置好的 PyTorch。课程中的两个 LoRA 模型通过固定种子构造并冻结，没有下载和微调动作。第 1—9 节关注指标和契约，第 10—15 节关注真实 KV 模型、SLO 调度和演练，第 16—18 节核对结果、检查和独立练习。

脚本支持：

```bash
python lesson_76_slo_tail_latency_admission_fairness.py --checks-only
python lesson_76_slo_tail_latency_admission_fairness.py --math-only
python lesson_76_slo_tail_latency_admission_fairness.py
python lesson_76_slo_tail_latency_admission_fairness.py --reload-run lesson_76_outputs/实际运行目录
```

运行默认在当前目录新建 `lesson_76_outputs/run_时间戳/`；从不覆盖已有目录。内置练习默认关闭，先自己写再开检查。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第76课：推理SLO、尾延迟、准入控制与租户公平。

独立离线PyTorch教学实验。真实微型因果Transformer/LoRA在物理KV张量里增量读写，
与全上下文前向核对；在真实已计算KV页上实现完整块前缀复用。
没有PagedAttention融合GPU内核、真实GPU性能数据或生产级服务并发。
"""
from __future__ import annotations

# 环境、确定性身份与范围
import argparse
import copy
import hashlib
import json
import math
import platform
import random
import re
import tempfile
from collections import Counter, OrderedDict
import secrets
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要已有PyTorch环境；本课不自动安装或下载模型。") from exc

VERSION = "lesson76-slo-admission-v1"
SOURCES = {
    "pagedattention": "https://arxiv.org/abs/2309.06180",
    "transformers_cache": "https://huggingface.co/docs/transformers/main/en/cache_explanation",
    "pytorch_attention": "https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html",
    "pytorch_inference": "https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html",
    "reproducibility": "https://docs.pytorch.org/docs/stable/notes/randomness.html",
    "vllm_prefix_cache": "https://docs.vllm.ai/en/latest/design/prefix_caching/",
    "vllm_security": "https://docs.vllm.ai/en/stable/usage/security/",
    "vllm_internals": "https://vllm.ai/blog/2025-09-05-anatomy-of-vllm",
    "vllm_metrics": "https://github.com/vllm-project/vllm/blob/main/docs/design/metrics.md",
    "vllm_benchmark": "https://docs.vllm.ai/en/latest/benchmarking/cli/",
    "vllm_request_metrics": "https://docs.vllm.ai/en/latest/features/per_request_metrics/",
    "triton_batcher": "https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/batcher.html",
    "sre_overload": "https://sre.google/sre-book/handling-overload/",
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""

def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)

def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")

def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value

def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
seed_all(7601)
print('PyTorch:', torch.__version__)
print('CUDA可用:', torch.cuda.is_available())
print('本课默认CPU，无下载、无真实GPU时延测量')

PyTorch: 2.10.0+cpu
CUDA可用: False
本课默认CPU，无下载、无真实GPU时延测量


## 1. 指标口径先于 P99 数字

- **排队时间（queue wait）**：`admission_tick - arrival_tick`，仅有入场事件的请求有值。
- **TTFT（首 token 等待）**：`first_output_tick - arrival_tick`，仅曾实际输出首 token 的请求有值。
- **E2E（端到端完成）**：`end_tick - arrival_tick`，本课只对真正完成的请求统计其延迟；拒绝、超时不当作“零延迟完成”。
- **ITL（输出间隔）**：逐个相邻 token 事件的时间差，再按输出间隔聚合。
- **TPOT（每请求平均后续 token 时间）**：单个完成请求的 `(最后输出 tick-首输出 tick)/(输出数-1)`，单 token 响应没有TPOT样本。

各框架/测试客户端的 TTFT 起点可能不同。本课以**到达请求**为起点，vLLM 的客户端 bench TTFT 则以客户端发送到收到首输出为口径；要对照需先核对计时边界与单位。[vLLM Benchmark 定义](https://docs.vllm.ai/en/latest/benchmarking/cli/) · [vLLM per-request metrics](https://docs.vllm.ai/en/latest/features/per_request_metrics/)

这里每个已激活会话每 tick 最多输出一个 token，prefill 在准入 tick **不额外消耗逻辑时间**。因此模拟 ITL 为 1 tick 并不构成现实模型 decode 测速。

In [3]:
# 固定教学分词器与真实因果/LoRA模型（独立于前课文件）
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}

@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')

class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)

class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))

class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>=1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))

class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)

def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy

def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]

def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}

def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))

def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base

def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

In [4]:
print('时间分位数（nearest-rank）')
for p in (0.5,0.95,0.99):
    print('P', int(p*100), sorted([1,2,4,8,9])[math.ceil(p*5)-1])
print('P99在5个样本中恰好是最大值；不能当作有统计稳定性的尾延迟结论。')

时间分位数（nearest-rank）
P 50 4
P 95 9
P 99 9
P99在5个样本中恰好是最大值；不能当作有统计稳定性的尾延迟结论。


## 2. SLO 成功必须包括请求结果，而非只看已完成者

定义每个请求两个逻辑预算：`ttft_slo_ticks` 与 `e2e_slo_ticks`。一条请求仅当**正常完成**且两个时间都在阈值内，才计为 `timely_completed`。

本课同时记录：

\[
\mathrm{SLO\ attainment}_{all}=\frac{\#\mathrm{timely\ completed}}{\#\mathrm{all\ arrived}}
\]

\[
\mathrm{SLO\ attainment}_{conditional}=\frac{\#\mathrm{timely\ completed}}{\#\mathrm{completed}}
\]

第一项把排队拒绝、准入舍弃和超时都保留在**到达总数**里。第二项是诊断值，可能因为拒绝更多慢请求而提高，**不能代替第一项或凭此宣布服务变好**。空分母返回 `None`，不伪造成零或百分百。

SLO goodput 在此定义为 `及时完成请求数 / 实际模拟的逻辑 ticks`，不是 tokens/s、GPU 吞吐或商业承诺。Google SRE 的过载章节强调系统在超载时必须选择适当的限流与降级机制，而非无限积累请求。[Handling Overload](https://sre.google/sre-book/handling-overload/)

In [5]:
print('同样有3条合格响应：')
print('及时完成 / 全到达(3/8):', 3/8)
print('及时完成 / 仅已完成(3/3):', 3/3)
print('统计口径不同，不能仅展示100%那一行。')

同样有3条合格响应：
及时完成 / 全到达(3/8): 0.375
及时完成 / 仅已完成(3/3): 1.0
统计口径不同，不能仅展示100%那一行。


## 3. 先建立一个真实、可核对的模型与物理 KV 账本

与上一课保持连续：自建教学 byte-merge tokenizer、带 Chat Template 边界的提示、32 维两层因果 Transformer、两个可复现 LoRA 版本。原始文本中的字面控制符不作为结构 token。

真实 K/V 向量写入事先分配的 `torch.Tensor` 物理页。缓存完整前缀块时，根据**权重、LoRA、模板、租户可信共享范围和父前缀哈希链**控制复用；写时复制保证并行分叉不污染他人。

每个完成、取消、超时与拒绝场景都要走到唯一终态；清理后 KV 字节全部归零。前向一致性单独与完整因果模型核对，不能以“模型能输出”取代精度验证。这里没有融合 PagedAttention GPU 内核。[PagedAttention](https://arxiv.org/abs/2309.06180) · [vLLM 前缀缓存设计](https://docs.vllm.ai/en/latest/design/prefix_caching/)

In [6]:
# 同基座不同Adapter的只读模型版本
def build_models(seed: int=7201):
    """同一随机基座 + 两份独立Q/V LoRA。版本只是不可变教学对象。"""
    require(type(seed) is int and seed >= 0, 'seed非法')
    seed_all(seed)
    tok = ByteMergeChatTokenizer()
    cfg = ToyConfig(tok.vocab_size, dim=32, heads=4, layers=2, ff_dim=64, max_length=192)
    base = TinyChatLM(cfg)
    a = inject_lora(base, rank=4, alpha=8.0)
    b = inject_lora(base, rank=4, alpha=8.0)
    # B=0时LoRA作用为零；通过不同adapter B制造第二个可核对的版本。
    generator = torch.Generator().manual_seed(seed+1)
    with torch.no_grad():
        for _,param in adapters(b):
            if param is not None and param.ndim == 2 and param.shape[1] == 4:  # B:[out,rank]
                param.copy_(torch.randn(param.shape, generator=generator)*0.12)
    models={'v0001':a.requires_grad_(False).eval(),
            'v0002':b.requires_grad_(False).eval()}
    require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
            '服务模型必须全部冻结')
    require(base_hash(models['v0001']) == base_hash(models['v0002']),
            '两个版本必须来自同一基座')
    require(state_hash(models['v0001'].state_dict()) != state_hash(models['v0002'].state_dict()),
            '版本权重应该不同')
    return tok,models

In [7]:
tok,models=build_models(7601)
print('词表大小:',tok.vocab_size)
print('模型版本:',list(models))
print('基座/LoRA总参数:',{k:sum(p.numel() for p in m.parameters()) for k,m in models.items()})
print('推理参数全部冻结:',all(not p.requires_grad for m in models.values() for p in m.parameters()))

词表大小: 281
模型版本: ['v0001', 'v0002']
基座/LoRA总参数: {'v0001': 42048, 'v0002': 42048}
推理参数全部冻结: True


## 4. 物理 KV 页池：有限块数不能在指标表中“超卖”

以下基础实现包含物理 KV 张量、块表、引用计数、容量检查与回滚；没有一个把 KV 消耗伪装成“仅 Python 计数”的捷径。该代码单元内容较长，可折叠后继续阅读。

需要区分：**请求入场的最大未来页预留**与**此刻真实占用页**。本课调度器对未来 decode 长度预检，但物理内存始终由页池核算；检查它们是否相容。

In [8]:
# 实体KV页池：容量、块表、引用计数与回滚
@dataclass
class KVSequence:
    version: str
    tokens: list[int] = field(default_factory=list)
    block_table: list[int] = field(default_factory=list)
    pending: dict | None = None


class NoKVSpace(LessonError):
    """资源耗尽是明确失败，不能当成EOS、正常输出或偷偷丢弃KV。"""


class PagedKVPool:
    """实际CPU Tensor承载所有layer的K/V；块表是教学页索引而非vLLM kernel。

    backing: [block, layer, key_or_value, slot, kv_head, head_dim]。
    模型和版本不可在已有序列上切换；共享块只在相同版本下合法。
    尾块写入时若被共享，先Copy-on-Write复制整个块所有层再写新token。
    """
    def __init__(self, capacity_blocks: int, block_size: int, model_cfg: ToyConfig,
                 versions: set[str], dtype: torch.dtype=torch.float32):
        require(type(capacity_blocks) is int and capacity_blocks>0,'块容量必须是正整数')
        require(type(block_size) is int and block_size>0,'块token大小必须是正整数')
        model_cfg.validate()
        require(type(versions) is set and bool(versions) and all(type(v) is str and v for v in versions),
                '版本集合非法')
        require(dtype in {torch.float32,torch.float64},'教学池仅实现float32/float64')
        self.cfg=model_cfg;self.capacity_blocks=capacity_blocks;self.block_size=block_size
        self.versions=set(versions);self.dtype=dtype
        self.head_dim=model_cfg.dim//model_cfg.heads
        self.data=torch.zeros((capacity_blocks,model_cfg.layers,2,block_size,
                               model_cfg.heads,self.head_dim),dtype=dtype,device='cpu')
        self.refs=[0]*capacity_blocks
        self.free=list(range(capacity_blocks))
        self.sessions:dict[str,KVSequence]={}
        self.allocation_count=0;self.reuse_count=0;self.cow_count=0
        self.oom_count=0;self.peak_live_blocks=0;self.peak_sessions=0
        self.zeroized_releases=0

    @property
    def bytes_per_token(self):
        return 2*self.cfg.layers*self.cfg.heads*self.head_dim*self.data.element_size()

    @property
    def bytes_per_block(self):
        return self.bytes_per_token*self.block_size

    def _sid(self,sid):
        bounded_str(sid,48,'会话ID')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',sid)),'会话ID仅限字母数字_-')
        return sid

    def _alloc(self):
        if not self.free:
            self.oom_count+=1
            raise NoKVSpace('KV页池容量不足；拒绝本次操作且不静默覆盖缓存')
        bid=self.free.pop(0)
        require(self.refs[bid]==0,'空闲块引用数不为0')
        require(not bool(self.data[bid].count_nonzero()),'空闲块残留数据，拒绝泄漏')
        self.refs[bid]=1;self.allocation_count+=1
        self.peak_live_blocks=max(self.peak_live_blocks,self.capacity_blocks-len(self.free))
        return bid

    def _decref(self,bid):
        require(type(bid) is int and 0<=bid<self.capacity_blocks and self.refs[bid]>0,
                '引用计数下溢或块ID非法')
        self.refs[bid]-=1
        if self.refs[bid]==0:
            with torch.no_grad():self.data[bid].zero_()
            self.free.append(bid);self.free.sort();self.zeroized_releases+=1

    def open(self,sid:str,version:str):
        self._sid(sid)
        require(version in self.versions,'未知模型版本不能建缓存')
        require(sid not in self.sessions,'不能重复建同一会话')
        self.sessions[sid]=KVSequence(version=version)
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))

    def fork(self,parent:str,child:str,version:str):
        self._sid(child)
        require(parent in self.sessions and child not in self.sessions,'fork父会话缺失或子会话已存在')
        p=self.sessions[parent]
        require(p.pending is None,'不能复制正在写入的缓存')
        require(version==p.version,'跨模型版本不能共享KV：即使基座相同adapter也可能不同')
        for bid in p.block_table:self.refs[bid]+=1
        self.sessions[child]=KVSequence(version=version,tokens=p.tokens.copy(),
                                        block_table=p.block_table.copy())
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))
        self.audit()

    def begin(self,sid:str,token_id:int,max_length:int):
        require(sid in self.sessions,'不存在的会话')
        s=self.sessions[sid]
        require(s.pending is None,'同一会话不允许并发写入')
        require(type(token_id) is int and token_id>=0,'新token ID非法')
        require(type(max_length) is int and len(s.tokens)<max_length,'超过上下文长度')
        slot=len(s.tokens)%self.block_size
        new_page=False;cow_old=None
        # 先检查容量，错误时保留全部原有块表和引用数。
        if slot==0 or (slot!=0 and self.refs[s.block_table[-1]]>1):
            if not self.free:
                self.oom_count+=1
                raise NoKVSpace('新页或写时复制需要空闲块，但KV页池已满')
            new_page=True
        if new_page:
            bid=self._alloc()
            if slot==0:
                s.block_table.append(bid)
            else:
                cow_old=s.block_table[-1]
                with torch.no_grad():self.data[bid].copy_(self.data[cow_old])
                self._decref(cow_old)
                s.block_table[-1]=bid
                self.cow_count+=1
        else:
            bid=s.block_table[-1]
        s.pending={'token_id':token_id,'slot':slot,'block':bid,
                   'new_page':new_page,'cow_old':cow_old,'written':set()}

    def put_layer(self,sid:str,layer:int,k:torch.Tensor,v:torch.Tensor):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'没有开始token写入')
        p=self.sessions[sid].pending
        require(type(layer) is int and 0<=layer<self.cfg.layers,'层ID非法')
        require(layer not in p['written'],'同一层重复写入')
        expected=(self.cfg.heads,self.head_dim)
        require(k.shape==expected and v.shape==expected and k.dtype==self.dtype and v.dtype==self.dtype,
                'K/V形状或dtype非法')
        require(k.device.type=='cpu' and v.device.type=='cpu','此教学页池只接受CPU K/V')
        require(bool(torch.isfinite(k).all()) and bool(torch.isfinite(v).all()),'K/V有非有限值')
        with torch.no_grad():
            self.data[p['block'],layer,0,p['slot']].copy_(k.detach())
            self.data[p['block'],layer,1,p['slot']].copy_(v.detach())
        p['written'].add(layer)

    def read_layer(self,sid:str,layer:int) -> tuple[torch.Tensor,torch.Tensor]:
        require(sid in self.sessions and self.sessions[sid].pending is not None,'read必须在token事务中')
        s=self.sessions[sid];p=s.pending
        require(layer in p['written'],'只能读取已写入当前token的层')
        length=len(s.tokens)+1
        ks=[];vs=[]
        for index,bid in enumerate(s.block_table):
            n=min(self.block_size,length-index*self.block_size)
            if n<=0:break
            # backing的slot轴在前，需要转成[heads,time,head_dim]。
            ks.append(self.data[bid,layer,0,:n].transpose(0,1))
            vs.append(self.data[bid,layer,1,:n].transpose(0,1))
        require(bool(ks) and sum(x.shape[1] for x in ks)==length,'KV块表与长度不同步')
        return torch.cat(ks,dim=1),torch.cat(vs,dim=1)

    def commit(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可提交的token')
        s=self.sessions[sid];p=s.pending
        require(p['written']==set(range(self.cfg.layers)),'有层尚未写入：禁止提交不完整KV')
        s.tokens.append(p['token_id']);s.pending=None
        self.audit()

    def abort(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可回滚的token')
        s=self.sessions[sid];p=s.pending
        bid=p['block'];slot=p['slot']
        if p['new_page']:
            if p['cow_old'] is not None:
                old=p['cow_old']
                s.block_table[-1]=old
                self.refs[old]+=1
            else:
                require(s.block_table[-1]==bid,'回滚末尾块不一致')
                s.block_table.pop()
            self._decref(bid)
        else:
            # 当前slot仍不属于任何已提交token，清空可能部分写入的K/V。
            with torch.no_grad():self.data[bid,:,:,slot].zero_()
        s.pending=None
        self.audit()

    def release(self,sid):
        require(sid in self.sessions,'重复释放或会话不存在')
        s=self.sessions[sid]
        require(s.pending is None,'请先回滚未完成写入再释放')
        for bid in s.block_table:self._decref(bid)
        del self.sessions[sid]
        self.audit()

    def usage(self) -> dict:
        table_ids=[b for s in self.sessions.values() for b in s.block_table]
        live=self.capacity_blocks-len(self.free)
        # 若block被两个会话引用，其有效token只计物理占用一次。
        filled={}
        for s in self.sessions.values():
            length=len(s.tokens)+(1 if s.pending else 0)
            for i,bid in enumerate(s.block_table):
                filled[bid]=max(filled.get(bid,0),min(self.block_size,length-i*self.block_size))
        occupied=sum(filled.values())
        logical_refs=len(table_ids)
        return {'capacity_blocks':self.capacity_blocks,'block_size':self.block_size,
                'free_blocks':len(self.free),'live_blocks':live,
                'physical_storage_bytes':self.data.numel()*self.data.element_size(),
                'bytes_per_token':self.bytes_per_token,'bytes_per_block':self.bytes_per_block,
                'live_reserved_bytes':live*self.bytes_per_block,
                'physical_used_slots':occupied,'live_content_bytes':occupied*self.bytes_per_token,
                'physical_internal_fragmentation_bytes':(live*self.block_size-occupied)*self.bytes_per_token,
                'logical_token_uses':sum(len(s.tokens) for s in self.sessions.values()),
                'logical_page_references':logical_refs,
                'saved_physical_blocks_by_sharing':logical_refs-live,
                'session_count':len(self.sessions),'peak_live_blocks':self.peak_live_blocks,
                'peak_sessions':self.peak_sessions,'allocation_count':self.allocation_count,
                'cow_count':self.cow_count,'oom_count':self.oom_count,
                'zeroized_releases':self.zeroized_releases}

    def audit(self) -> dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话绑定未知版本')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '块表页数与有效上下文长度不一致')
            require(len(set(s.block_table))==len(s.block_table),'同一会话不能重复引用物理块')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'块表有非法物理页')
                expected[bid]+=1
        require(expected==self.refs,'物理块引用计数不等于块表的真实引用')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free list/引用计数不一致')
        require(len(self.free)==len(set(self.free)),'空闲页重复入队')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),
                '空闲块包含未清除的KV数据')
        u=self.usage()
        require(u['physical_internal_fragmentation_bytes']>=0,'内部碎片负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '实际/保留字节账目不守恒')
        return u

## 5. 同一模型上的增量 decode 和完整前向必须一致

增量执行器每步只对新增位置算注意力，实际读取此前保存在 KV 页中的 K/V。最终 logits 与在同样前缀上做完整前向的 logits 比较，容差来自有限精度运算，不是语义等价猜测。

In [9]:
# 增量模型执行器：Q/K/V真实写入、块表聚合与全上下文比较
class CachedModelRunner:
    """一次处理一个新增token。prefill本课逐token执行，非生产高吞吐chunked prefill。

    同一层先写该token的K/V，再用块表读取全部历史K/V，对单query计算attention。
    该前缀计算和原模型full forward的掩码、位置、LoRA权重完全一致。
    """
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],pool:PagedKVPool):
        require(set(models)==pool.versions,'模型版本与池的版本集合不一致')
        require(all(m.cfg==pool.cfg for m in models.values()),'模型配置和池配置不匹配')
        require(pool.dtype==torch.float32,'本课模型采用float32，KV存储dtype必须一致')
        require(all(next(m.parameters()).device.type=='cpu' for m in models.values()),'教学KV池仅支持CPU')
        require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
                'KV缓存仅用于冻结推理模型，不能偷偷反传训练')
        self.tok=tok;self.models=models;self.pool=pool
        self.last_logits:dict[str,torch.Tensor]={}
        self.tokens_processed=0;self.attention_query_key_products=0
        self.full_recompute_query_key_products=0
        self.model_hashes={name:state_hash(m.state_dict()) for name,m in models.items()}

    @torch.inference_mode()
    def step(self,sid:str,token_id:int) -> torch.Tensor:
        require(sid in self.pool.sessions,'cache step会话不存在')
        s=self.pool.sessions[sid]
        require(type(token_id) is int and 0<=token_id<self.pool.cfg.vocab_size,'token ID非法')
        length=len(s.tokens)
        require(length<self.pool.cfg.max_length,'上下文长度已满')
        self.pool.begin(sid,token_id,self.pool.cfg.max_length)
        model=self.models[s.version]
        try:
            ids=torch.tensor([[token_id]],dtype=torch.long)
            pos=torch.tensor([length],dtype=torch.long)
            x=model.embed(ids)+model.position(pos)[None]
            heads=self.pool.cfg.heads;head_dim=self.pool.head_dim
            for index,block in enumerate(model.blocks):
                z=block.norm1(x)
                q=block.attn.q_proj(z).reshape(heads,head_dim)
                k=block.attn.k_proj(z).reshape(heads,head_dim)
                v=block.attn.v_proj(z).reshape(heads,head_dim)
                self.pool.put_layer(sid,index,k,v)
                all_k,all_v=self.pool.read_layer(sid,index)
                # query:[heads,1,d]; keys:[heads,seq,d]；此query位于整个缓存最末尾，
                # 所以其可见范围恰好是全部已存的过去token与当前token。
                scores=(q[:,None,:]@all_k.transpose(-2,-1))/math.sqrt(head_dim)
                weights=torch.softmax(scores,dim=-1)
                value=(weights@all_v).reshape(1,1,self.pool.cfg.dim)
                x=x+block.attn.o_proj(value)
                x=x+block.ff(block.norm2(x))
                self.attention_query_key_products+=heads*(length+1)
                self.full_recompute_query_key_products+=heads*(length+1)**2
            next_logits=model.head(model.norm(x))[0,0].detach().clone()
            require(bool(torch.isfinite(next_logits).all()),'模型输出非有限logits')
            self.pool.commit(sid)
            self.last_logits[sid]=next_logits
            self.tokens_processed+=1
            return next_logits
        except Exception:
            if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
            raise

    def prefill(self,sid:str,ids:list[int]) -> torch.Tensor:
        require(type(ids) is list and bool(ids),'prefill需要非空token列表')
        require(all(type(i) is int and 0<=i<self.pool.cfg.vocab_size for i in ids),
                'prefill出现非法token')
        require(sid in self.pool.sessions,'prefill会话不存在')
        require(len(self.pool.sessions[sid].tokens)+len(ids)<=self.pool.cfg.max_length,
                'prefill不能越界或静默截断')
        last=None
        for token in ids:last=self.step(sid,token)
        return last

    def open_prefill(self,sid,version,ids):
        self.pool.open(sid,version)
        try:return self.prefill(sid,ids)
        except Exception:
            self.close(sid)
            raise

    def fork(self,parent,child,version):
        require(parent in self.last_logits,'父序列尚未prefill，不能fork')
        self.pool.fork(parent,child,version)
        self.last_logits[child]=self.last_logits[parent].clone()

    def close(self,sid):
        self.pool.release(sid)
        self.last_logits.pop(sid,None)

    @torch.inference_mode()
    def full_last(self,version:str,ids:list[int]) -> torch.Tensor:
        require(version in self.models and bool(ids),'版本或输入非法')
        model=self.models[version]
        inp=torch.tensor([ids],dtype=torch.long)
        mask=torch.ones_like(inp,dtype=torch.bool)
        return model(inp,mask)[0,-1].detach().clone()

    def assert_consistent(self,sid,atol=2e-5):
        require(sid in self.pool.sessions and sid in self.last_logits,'尚无缓存输出')
        s=self.pool.sessions[sid]
        expected=self.full_last(s.version,s.tokens)
        got=self.last_logits[sid]
        max_delta=float((expected-got).abs().max())
        require(torch.allclose(expected,got,atol=atol,rtol=2e-5),
                '真实cache输出与full recompute不一致：检查位置、KV存储或因果掩码')
        return max_delta

## 6. 缓存身份与租户：命中率不是隔离证明

两个主体只有在可信权限配置允许共享时才能命中同一前缀。不同模型、LoRA、模板、媒体摘要都必须改变缓存身份。新 SLO 策略只能修改准入次序，不能绕过这些身份检查。

策略提前拒绝的请求**根本没有执行 prefill**，不能把它的 token 数加入“已准入提示 token 命中率”的分母；同理，不能让它从全到达请求的 SLO 成功率分母里消失。

In [10]:
# 可信共享范围、模型/模板/媒体身份与完整块哈希链
@dataclass(frozen=True)
class CacheRequest:
    """教学内部调用：principal必须由可信网关鉴权后传入，不能让客户端自行声明。"""
    sid: str
    principal: str
    version: str
    messages: tuple[tuple[str,str], ...]
    template_revision: str = "chat-v1"
    media_bytes: bytes | None = None
    add_generation_prompt: bool = True


class TrustedScopeAuthority:
    """服务器控制隔离组与加密随机盐，不接受请求自行提供salt。

    本地示例没有HTTP/登录鉴权。真实调用者必须由网关先确认principal身份，
    本类不防御攻击者能直接调用内部Python对象的情形。
    """
    def __init__(self, principal_groups: dict[str,str]):
        require(type(principal_groups) is dict and bool(principal_groups), '需要非空可信身份组')
        require(all(type(k) is str and k and type(v) is str and v
                    for k,v in principal_groups.items()),'可信身份/共享组非法')
        self._groups=principal_groups.copy()
        self._secrets={g:secrets.token_bytes(32) for g in set(principal_groups.values())}

    def group(self,principal: str) -> str:
        require(type(principal) is str and principal in self._groups,
                '未知或未认证主体不能使用Prefix Cache')
        return self._groups[principal]

    def fingerprint(self,principal: str) -> str:
        group=self.group(principal)
        return hashlib.sha256(self._secrets[group]).hexdigest()

    def rotate(self,group: str) -> str:
        require(group in self._secrets, '未知共享组')
        old=hashlib.sha256(self._secrets[group]).hexdigest()
        self._secrets[group]=secrets.token_bytes(32)
        require(self.fingerprint(next(k for k,v in self._groups.items() if v==group))!=old,
                '加密随机盐轮换未生效')
        return old

    def shared(self,p1:str,p2:str)->bool:
        return self.group(p1)==self.group(p2)


@dataclass(frozen=True)
class CacheIdentity:
    version: str
    model_hash: str
    base_hash: str
    tokenizer_hash: str
    template_hash: str
    scope_hash: str
    media_hash: str
    compute_hash: str


def request_tokens(tok:ByteMergeChatTokenizer,req:CacheRequest)->list[int]:
    require(isinstance(req,CacheRequest),'必须传入CacheRequest')
    require(type(req.messages) is tuple and bool(req.messages),'消息必须为非空tuple')
    require(type(req.add_generation_prompt) is bool,'生成提示选项非法')
    bounded_str(req.template_revision,64,'模板修订号')
    require(bool(req.template_revision),'模板修订号不能为空')
    messages=[]
    for item in req.messages:
        require(type(item) is tuple and len(item)==2 and item[0] in tok.role_ids
                and type(item[1]) is str,'消息必须为(role,content)')
        messages.append({'role':item[0],'content':item[1]})
    return tok.apply_chat_template(messages,add_generation_prompt=req.add_generation_prompt)


def make_identity(req:CacheRequest,authority:TrustedScopeAuthority,tok:ByteMergeChatTokenizer,
                  models:dict[str,TinyChatLM],saved_hashes:dict[str,str]) -> CacheIdentity:
    require(req.version in models and req.version in saved_hashes,'版本没有被可信模型库登记')
    authority.group(req.principal)
    require(type(req.media_bytes) in (bytes,type(None)),'媒体身份必须取自受信预处理的原始字节')
    require(req.media_bytes is None or len(req.media_bytes)<=4096,
            '教学元数据媒体上限4096字节；不允许超大请求键')
    # 禁止在同一个版本名称下原地修改模型而继续读取旧KV。
    model=models[req.version]
    current=state_hash(model.state_dict())
    require(current==saved_hashes[req.version], '热更新必须显式替换并失效缓存，不能原地修改模型')
    return CacheIdentity(
        version=req.version,
        model_hash=current,
        base_hash=base_hash(model),
        tokenizer_hash=digest(tok.contract()),
        template_hash=digest({'family':'toy-message-boundary','revision':req.template_revision,
                              'generation_prompt':req.add_generation_prompt,
                              'role_end':'assistant=eos,others=im_end',
                              'tokenizer':tok.contract()}),
        scope_hash=authority.fingerprint(req.principal),
        media_hash=('none' if req.media_bytes is None else
                    hashlib.sha256(req.media_bytes).hexdigest()),
        compute_hash=digest({'cfg':asdict(model.cfg),'dtype':str(next(model.parameters()).dtype),
                             'position':'absolute','attention':'full causal',
                             'kv_layout':'blocks,layer,kv,slot,heads,dim',
                             'runtime':'toy-cpu-fp32-v1'}),
    )


def prefix_keys(identity:CacheIdentity,ids:list[int],block_size:int)->list[str]:
    require(isinstance(identity,CacheIdentity),'缺少可信缓存身份')
    require(type(block_size) is int and block_size>0,'block_size必须为正整数')
    require(type(ids) is list and all(type(t) is int and t>=0 for t in ids),'token列表非法')
    parent=digest({'kind':'scoped-prefix-root','version':VERSION,'identity':asdict(identity)})
    keys=[]
    for start in range(0,len(ids)//block_size*block_size,block_size):
        block=ids[start:start+block_size]
        parent=digest({'kind':'full-prefix-block','parent_hash':parent,
                       'position':start,'token_ids':block})
        keys.append(parent)
    return keys


@dataclass
class CachedBlock:
    key: str
    physical_page: int
    identity: CacheIdentity
    prefix_tokens: tuple[int,...]   # 仅在可信进程内保留，用于哈希碰撞检测；报告禁止展示
    block_tokens: tuple[int,...]
    parent_key: str | None
    end_logits: torch.Tensor
    kv_hash: str
    logits_hash: str
    last_access_tick: int

In [11]:
# 对真实KV页加缓存所有权：命中、LRU、失效、物理回收
class PrefixKVPool(PagedKVPool):
    """缓存pin与活动会话引用共同拥有物理页。只发布已提交的完整块。

    与工业vLLM不同：CPU同步、预分配Tensor、无真实异步锁、无融合页注意力核。
    精确前缀tokens用于碰撞检测，存储成本未计入KV字节预算。
    """
    def __init__(self,capacity_blocks:int,block_size:int,model_cfg:ToyConfig,versions:set[str],
                 cache_capacity_blocks:int|None=None):
        super().__init__(capacity_blocks,block_size,model_cfg,versions)
        cache_capacity_blocks=capacity_blocks if cache_capacity_blocks is None else cache_capacity_blocks
        require(type(cache_capacity_blocks) is int and 0<=cache_capacity_blocks<=capacity_blocks,
                '缓存块上限不能超过物理容量')
        self.cache_capacity_blocks=cache_capacity_blocks
        self.cache:dict[str,CachedBlock]={}
        self.clock=0
        self.hit_blocks=0;self.hit_tokens=0;self.miss_requests=0
        self.promotions=0;self.evictions=0;self.invalidations=0
        self.skipped_promotions=0;self.zero_hit_requests=0

    def _tick(self):
        self.clock+=1
        return self.clock

    def _page_hash(self,bid:int)->str:
        return state_hash({'physical_KV':self.data[bid]})

    def _validate_entry(self,e:CachedBlock,identity:CacheIdentity,desired_prefix:tuple[int,...],
                        requested_key:str):
        require(e.key==requested_key and e.identity==identity and
                e.prefix_tokens==desired_prefix,'哈希碰撞或缓存身份/前缀不一致，必须拒绝命中')
        require(e.block_tokens==desired_prefix[-self.block_size:]
                and len(e.block_tokens)==self.block_size,'只能命中完整块')
        require(e.kv_hash==self._page_hash(e.physical_page),'缓存KV字节损坏，拒绝复用')
        require(e.logits_hash==state_hash({'end_logits':e.end_logits}),
                '缓存块末端logits字节损坏，拒绝复用')
        require(self.refs[e.physical_page]>=1,'缓存页已经释放，索引悬空')

    def lookup(self,identity:CacheIdentity,ids:list[int])->list[CachedBlock]:
        keys=prefix_keys(identity,ids,self.block_size)
        found=[]
        for i,key in enumerate(keys):
            e=self.cache.get(key)
            if e is None:break
            self._validate_entry(e,identity,tuple(ids[:(i+1)*self.block_size]),key)
            # 顺序访问同时更新LRU；匹配只返回连续前缀，不能跳过中间缺失块。
            e.last_access_tick=self._tick()
            found.append(e)
        return found

    def _evict_one_idle(self)->bool:
        choices=[e for e in self.cache.values() if self.refs[e.physical_page]==1]
        if not choices:return False
        victim=min(choices,key=lambda e:(e.last_access_tick,e.key))
        del self.cache[victim.key]
        self._decref(victim.physical_page)
        self.evictions+=1
        return True

    def _alloc(self):
        if not self.free:
            self._evict_one_idle()
        return super()._alloc()

    def begin(self,sid,token_id,max_length):
        # 父类在发现free为空时会先报OOM；尝试驱逐idle cache才进入它的begin。
        if sid in self.sessions:
            s=self.sessions[sid]
            if s.pending is None and not self.free:
                slot=len(s.tokens)%self.block_size
                needs_page=slot==0 or (slot>0 and self.refs[s.block_table[-1]]>1)
                if needs_page:self._evict_one_idle()
        return super().begin(sid,token_id,max_length)

    def publish(self,sid:str,index:int,key:str,identity:CacheIdentity,
                prefix_tokens:tuple[int,...],end_logits:torch.Tensor,
                parent_key:str|None)->bool:
        require(sid in self.sessions,'发布会话不存在')
        s=self.sessions[sid]
        require(s.pending is None and s.version==identity.version,'不允许发布未提交/跨版本KV')
        require(type(index) is int and 0<=index<len(s.block_table),'发布块下标错误')
        require(len(prefix_tokens)==(index+1)*self.block_size and
                tuple(s.tokens[:len(prefix_tokens)])==prefix_tokens,'前缀token身份错误')
        require(end_logits.shape==(self.cfg.vocab_size,) and end_logits.dtype==self.dtype and
                bool(torch.isfinite(end_logits).all()),'末端logits形状/数值错误')
        chain=prefix_keys(identity,list(prefix_tokens),self.block_size)
        require(chain and chain[-1]==key and (chain[-2] if len(chain)>1 else None)==parent_key,
                '发布哈希链与token前缀不一致')
        if key in self.cache:
            self._validate_entry(self.cache[key],identity,prefix_tokens,key)
            return False
        if self.cache_capacity_blocks==0:
            self.skipped_promotions+=1
            return False
        if len(self.cache)>=self.cache_capacity_blocks and not self._evict_one_idle():
            self.skipped_promotions+=1
            return False
        bid=s.block_table[index]
        require(self.refs[bid]>=1,'发布物理页没有会话所有权')
        self.refs[bid]+=1             # 独立cache pin：请求结束后仍可命中
        logit=end_logits.detach().cpu().contiguous().clone()
        self.cache[key]=CachedBlock(key=key,physical_page=bid,identity=identity,
             prefix_tokens=prefix_tokens,block_tokens=prefix_tokens[-self.block_size:],
             parent_key=parent_key,end_logits=logit,kv_hash=self._page_hash(bid),
             logits_hash=state_hash({'end_logits':logit}),last_access_tick=self._tick())
        self.promotions+=1
        self.audit()
        return True

    def invalidate(self,predicate:Callable[[CachedBlock],bool]|None=None)->int:
        """移除pin。即使仍有活动会话，只有它们的引用会保留KV。"""
        selected=[key for key,e in self.cache.items() if predicate is None or predicate(e)]
        for key in selected:
            e=self.cache.pop(key)
            self._decref(e.physical_page)
        self.invalidations+=len(selected)
        self.audit()
        return len(selected)

    def advance_clock_and_expire(self,ticks:int,max_idle_ticks:int)->int:
        require(type(ticks) is int and ticks>=0 and type(max_idle_ticks) is int and max_idle_ticks>=0,
                '逻辑时间参数非法')
        self.clock+=ticks
        return self.invalidate(lambda e: self.clock-e.last_access_tick>max_idle_ticks)

    def cache_usage(self)->dict:
        total=self.usage()
        idle_cached=sum(self.refs[e.physical_page]==1 for e in self.cache.values())
        return {'cache_blocks':len(self.cache),'cache_capacity_blocks':self.cache_capacity_blocks,
                'idle_cached_blocks':idle_cached,'active_cached_blocks':len(self.cache)-idle_cached,
                'cache_pins':len(self.cache),'cache_held_bytes':len(self.cache)*self.bytes_per_block,
                'hit_blocks':self.hit_blocks,'hit_tokens':self.hit_tokens,
                'promotions':self.promotions,'evictions':self.evictions,
                'invalidations':self.invalidations,'skipped_promotions':self.skipped_promotions,
                'live_blocks':total['live_blocks'],'free_blocks':total['free_blocks'],
                'session_count':len(self.sessions),'physical_storage_bytes':total['physical_storage_bytes']}

    def usage(self)->dict:
        result=super().usage()
        result['cache_pins']=len(self.cache)
        result['cache_held_bytes']=len(self.cache)*self.bytes_per_block
        # 物理页可以只由缓存持有，因此旧课的logical_refs-live会变负：需把pin计入。
        result['saved_physical_blocks_by_sharing']=(result['logical_page_references']+
                                                     len(self.cache)-result['live_blocks'])
        return result

    def audit(self)->dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话模型版本不在白名单')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '会话页数与token长度不匹配')
            require(len(s.block_table)==len(set(s.block_table)),'同一会话重复引用页')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'页ID越界')
                expected[bid]+=1
        cache_pages=set()
        for key,e in self.cache.items():
            require(key==e.key and e.physical_page not in cache_pages,'重复pin或缓存键/页错配')
            cache_pages.add(e.physical_page)
            require(0<=e.physical_page<self.capacity_blocks,'缓存物理页非法')
            require(len(e.prefix_tokens)%self.block_size==0 and len(e.block_tokens)==self.block_size,
                    '缓存只允许完整块')
            expected[e.physical_page]+=1
        require(expected==self.refs,'会话引用+缓存pin不等于实际物理引用数')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free-list不守恒')
        require(len(set(self.free))==len(self.free),'free-list有重复')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),'空闲页未清零')
        u=self.usage()
        require(u['saved_physical_blocks_by_sharing']>=0,'物理共享账本负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '字节账本不守恒')
        return u

## 7. 命中后也要执行剩余真实 prefill

前缀缓存重用的是完整块中真正计算过的 K/V，不是“预测可以命中所以跳过模型”。未命中的后续提示仍要计算，模型版本必须与缓存一致。每个场景都要满足：

```text
真实新计算的prefill token + 真实重用的prefill token = 已准入的全部prompt token
```

In [12]:
# 前缀命中后继续真实预填充，并与完整因果前向核对
class PrefixCacheRunner:
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],
                 pool:PrefixKVPool,authority:TrustedScopeAuthority):
        require(isinstance(pool,PrefixKVPool),'需要有缓存所有权的KV页池')
        self.tok=tok;self.models=models;self.pool=pool;self.authority=authority
        self.runner=CachedModelRunner(tok,models,pool)
        self.bound:dict[str,CacheIdentity]={}
        self.events:list[dict]=[]

    def admit(self,req:CacheRequest)->dict:
        require(isinstance(req,CacheRequest),'请求类型非法')
        self.pool._sid(req.sid)
        require(req.sid not in self.pool.sessions,'sid不可重复入场')
        # 注意：principal须来自已鉴权的内部调用上下文，不应接受HTTP客户端自报。
        ids=request_tokens(self.tok,req)
        require(bool(ids) and len(ids)<=self.pool.cfg.max_length,'提示过长或为空')
        ident=make_identity(req,self.authority,self.tok,self.models,self.runner.model_hashes)
        hits=self.pool.lookup(ident,ids)
        hit_count=len(hits)*self.pool.block_size
        keys=prefix_keys(ident,ids,self.pool.block_size)
        self.pool.open(req.sid,req.version)
        self.bound[req.sid]=ident
        end_logits:dict[int,torch.Tensor]={}
        try:
            state=self.pool.sessions[req.sid]
            for i,e in enumerate(hits):
                self.pool.refs[e.physical_page]+=1
                state.block_table.append(e.physical_page)
                state.tokens.extend(e.block_tokens)
                end_logits[i]=e.end_logits.detach().clone()
            if hits:
                self.runner.last_logits[req.sid]=hits[-1].end_logits.detach().clone()
            self.pool.audit()
            for t in ids[hit_count:]:
                out=self.runner.step(req.sid,t)
                n=len(self.pool.sessions[req.sid].tokens)
                if n%self.pool.block_size==0:
                    end_logits[n//self.pool.block_size-1]=out.detach().clone()
            require(self.pool.sessions[req.sid].tokens==ids,'前缀命中恢复后的token序列错误')
            require(req.sid in self.runner.last_logits,'请求没有任何合法预测logits')
            # 先计算整个prompt，再发布完整块；失败入场不残留不完整页索引。
            parent=None
            for i,key in enumerate(keys):
                prefix=tuple(ids[:(i+1)*self.pool.block_size])
                require(i in end_logits,'完整块末端logits没有计算/回填')
                self.pool.publish(req.sid,i,key,ident,prefix,end_logits[i],parent)
                parent=key
            self.pool.hit_blocks+=len(hits);self.pool.hit_tokens+=hit_count
            if not hits:self.pool.zero_hit_requests+=1
            max_delta=self.runner.assert_consistent(req.sid)
            result={'sid':req.sid,'version':req.version,'prompt_tokens':len(ids),
                    'hit_blocks':len(hits),'hit_tokens':hit_count,'new_tokens_computed':len(ids)-hit_count,
                    'max_logits_abs_delta':max_delta,'cache_blocks_after':len(self.pool.cache),
                    'scope':'internal_authenticated_principal'}
            self.events.append(result.copy())
            return result
        except Exception:
            # 失败不留下活动会话/泄漏物理页；不回滚之前已发生的idle LRU驱逐。
            if req.sid in self.pool.sessions:
                self.runner.close(req.sid)
            self.bound.pop(req.sid,None)
            self.pool.audit()
            raise

    def continue_token(self,sid:str,token_id:int)->float:
        require(sid in self.bound and sid in self.pool.sessions,'不存在的会话')
        identity=self.bound[sid]
        require(identity.model_hash==state_hash(self.models[identity.version].state_dict()),
                '活跃会话期间模型权重不可原地热修改')
        self.runner.step(sid,token_id)
        return self.runner.assert_consistent(sid)

    def fork_session(self,parent:str,child:str,principal:str):
        require(parent in self.bound and parent in self.pool.sessions,'父会话不存在')
        require(child not in self.bound and child not in self.pool.sessions,'子会话已经存在')
        identity=self.bound[parent]
        require(self.authority.fingerprint(principal)==identity.scope_hash,
                'fork只允许相同可信缓存共享范围，不能跨租户')
        self.runner.fork(parent,child,identity.version)
        self.bound[child]=identity
        self.pool.audit()

    def close(self,sid:str):
        require(sid in self.bound,'未授权/已经关闭的会话')
        self.runner.close(sid)
        self.bound.pop(sid)
        self.pool.audit()

    def invalidate_version(self,version:str)->int:
        require(version in self.models,'未知版本')
        return self.pool.invalidate(lambda e:e.identity.version==version)

    def rotate_group(self,group:str)->int:
        old_fp=self.authority.rotate(group)
        return self.pool.invalidate(lambda e:e.identity.scope_hash==old_fp)

    def replace_version(self,version:str,new_model:TinyChatLM)->int:
        require(version in self.models,'未知版本不能替换')
        require(all(s.version!=version for s in self.pool.sessions.values()),
                '有该版本的活跃会话，禁止原地替换/撕裂读写')
        require(isinstance(new_model,TinyChatLM) and new_model.cfg==self.pool.cfg,
                '新模型配置或类型错误')
        require(next(new_model.parameters()).device.type=='cpu'
                and next(new_model.parameters()).dtype==torch.float32,
                '新模型必须是CPU FP32')
        require(all(not p.requires_grad for p in new_model.parameters()),'新模型必须全部冻结')
        replacement=copy.deepcopy(new_model).eval()
        invalidated=self.invalidate_version(version)
        self.models[version]=replacement
        self.runner.models[version]=replacement
        self.runner.model_hashes[version]=state_hash(replacement.state_dict())
        return invalidated

    def final_flush(self)->dict:
        require(not self.pool.sessions and not self.bound,'flush前必须结束全部会话')
        self.pool.invalidate()
        usage=self.pool.audit()
        require(usage['live_blocks']==0 and not bool(self.pool.data.count_nonzero()),
                '全部缓存清空后仍有KV残留')
        return usage

## 8. 原调度器的限制：先有队列与资源，才谈新策略

基线调度器已经实现 FIFO 与租户轮转、活跃会话数、每租户并发限制、队列拒绝、真实 KV 容量预检、取消、等待超时、活跃超时，以及唯一终态。所有这些边界将原样保留。

关键时间顺序：同 tick 全部请求到达；检查取消和到期；已入场的会话产生下一个 token；然后从排队队列中准入。**当 tick=t 准入，最快在 t+1 输出第一个 token**。这个离散约定决定 SLO 的“等号边界”。

In [13]:
# 可复现的合成到达序列、容量配置与指标定义
@dataclass(frozen=True)
class TrafficRequest:
    sid: str
    arrival_tick: int
    principal: str
    body: str
    version: str = 'v0001'
    template_revision: str = 'chat-v1'
    patience_ticks: int = 16
    decode_steps: int = 2
    cancel_after_ticks: int | None = None

    def validate(self):
        bounded_str(self.sid, 48, 'sid')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+', self.sid)), 'sid字符非法')
        bounded_str(self.body, 512, '请求正文')
        require(self.principal in {'alice','bob','alice_worker','org1','org2'}, '请求主体必须来自可信测试名单')
        require(self.version in {'v0001','v0002'}, '未知模型版本')
        require(self.template_revision in {'chat-v1','chat-v2'}, '未知模板修订号')
        require(type(self.arrival_tick) is int and self.arrival_tick >= 0, '到达tick必须非负整数')
        require(type(self.patience_ticks) is int and self.patience_ticks > 0, '等待截止tick必须正整数')
        require(type(self.decode_steps) is int and 1 <= self.decode_steps <= 3, '生成预算1到3步')
        require(self.cancel_after_ticks is None or
                (type(self.cancel_after_ticks) is int and self.cancel_after_ticks >= 1),
                '主动取消必须是到达后的正整数tick')


@dataclass(frozen=True)
class LoadConfig:
    capacity_blocks: int = 64
    cache_limit: int = 24
    block_size: int = 4
    max_active: int = 3
    per_principal_active: int = 2
    queue_limit: int = 12
    max_ticks: int = 180
    policy: str = 'tenant_rr'  # FIFO严格队头，或租户RR跳过不可调度队头

    def validate(self):
        for k in ('capacity_blocks','block_size','max_active','per_principal_active',
                  'queue_limit','max_ticks'):
            v=getattr(self,k)
            require(type(v) is int and v > 0, k+'必须是正整数')
        require(type(self.cache_limit) is int and 0 <= self.cache_limit <= self.capacity_blocks,
                '缓存容量必须在0..物理页容量之间')
        require(self.per_principal_active <= self.max_active, '单租户并行度不能超过总并行度')
        require(self.policy in {'fifo','tenant_rr'}, '只支持严格FIFO/租户轮转')


def make_workload(kind: str, count: int=18) -> list[TrafficRequest]:
    """确定性的教学流量，不含真实用户请求；组内复用与跨租户请求交错。"""
    require(kind in {'steady_hot','burst_skew','isolation','thrash','cancel'}, '未知负载形态')
    require(type(count) is int and 2 <= count <= 120, '请求数量2..120')
    traffic=[]
    for i in range(count):
        if kind=='steady_hot':
            principal='alice' if i%4 else 'alice_worker'  # 可信同组，允许共享缓存
            arrival=3*i
            body='Hello world! '+('repeat' if i%5 else 'alternate')
        elif kind=='burst_skew':
            principal='bob' if i%6==0 else 'alice'
            arrival=(i//6)*2
            body='Hello world! '+('repeat' if i%3 else f'cold{i:02d}')
        elif kind=='isolation':
            principal='alice' if i%2==0 else 'bob'
            arrival=3*i
            body='Hello world! repeat'
        elif kind=='thrash':
            principal='alice'
            arrival=3*i
            body='Hello world! '+('repeat' if i%3 else f'unique-{i:02d}')
        else:
            principal='alice' if i%3 else 'bob'
            arrival=i//4
            body='Hello world! '+('repeat' if i%2 else 'alternate')
        traffic.append(TrafficRequest(sid=f'{kind[:3]}_{i:03d}',arrival_tick=arrival,
                          principal=principal,body=body,
                          version=('v0002' if kind=='isolation' and i%4==3 else 'v0001'),
                          patience_ticks=(5 if kind in {'burst_skew','cancel'} else 18),
                          decode_steps=2,
                          cancel_after_ticks=(2 if kind=='cancel' and i%7==0 else None)))
    validate_workload(traffic)
    return traffic


def validate_workload(items: list[TrafficRequest]) -> None:
    require(type(items) is list and bool(items), '需要非空流量')
    seen=set();last=-1
    for x in items:
        require(isinstance(x,TrafficRequest),'必须是TrafficRequest')
        x.validate()
        require(x.sid not in seen, '请求ID重复')
        require(x.arrival_tick>=last, '到达顺序必须是非降序；调用方显式排序')
        seen.add(x.sid);last=x.arrival_tick


def nearest_rank(data: list[int], p: float) -> int | None:
    """经验分位数nearest-rank：第ceil(p*n)个，空数据返回None。"""
    require(0 < p <= 1 and math.isfinite(p), '分位数p必须(0,1]')
    if not data:return None
    require(all(type(x) is int and x>=0 for x in data), '必须是非负整数的逻辑tick')
    ordered=sorted(data)
    return ordered[math.ceil(p*len(data))-1]


def metric_ratio(numerator: int, denominator: int) -> float | None:
    require(type(numerator) is int and type(denominator) is int and
            0 <= numerator <= denominator, '计数器不符合比例边界')
    return numerator/denominator if denominator else None


def jain_index(values: list[float]) -> float | None:
    """对各主体完成/到达比例算Jain，所有比例0则不定义；不能取代按主体报告。"""
    require(bool(values) and all(isinstance(x,(float,int)) and math.isfinite(x) and x>=0 for x in values),
            'Jain输入需要有限非负数')
    total=sum(values);denom=sum(x*x for x in values)
    return (total*total)/(len(values)*denom) if denom else None

In [14]:
# 物理KV约束下的调度器：明确队列、预检、租户限额、终态与统计
class LoadScheduler:
    """同步CPU按逻辑tick离散推进；一次入场做真实微型模型prefill。

    这不是GPU时间模型：每tick对每个活跃请求至多生成1token，prefill不占tick，
    模型实时计算花费的墙钟时间不写入仿真报告，也不据此声称TTFT/TPOT/goodput。
    """
    def __init__(self,service:PrefixCacheRunner,config:LoadConfig,items:list[TrafficRequest]):
        config.validate();validate_workload(items)
        require(items[-1].arrival_tick < config.max_ticks, '逻辑仿真窗口必须覆盖最后到达的请求')
        require(service.pool.capacity_blocks==config.capacity_blocks and
                service.pool.block_size==config.block_size and
                service.pool.cache_capacity_blocks==config.cache_limit,
                '负载容量配置和物理KV池不符')
        self.service=service;self.pool=service.pool;self.cfg=config
        self.items=items
        self.by_sid={r.sid:r for r in items}
        self.pending:list[str]=[]
        self.active:dict[str,dict]={}
        self.terminal:dict[str,dict]={}
        self.admitted:dict[str,dict]={}
        self.rr_tenants=sorted({r.principal for r in items})
        self.rr_next=0
        self.arrival_cursor=0
        self.capacity_deferrals=0;self.quota_deferrals=0
        self.slot_deferrals=0;self.queue_peak=0;self.peak_active=0
        self.prompts_computed=0;self.prompts_reused=0;self.total_prompt_tokens=0
        self.total_eligible_tokens=0;self.hit_requests=0
        self.peak_live_blocks=0;self.active_time_slots=0
        self.model_errors=0;self.finalized=False
        self.trace:list[dict]=[]
        self.rr_admit_order:list[str]=[]

    def _request(self,sid):
        r=self.by_sid[sid]
        return CacheRequest(sid=r.sid, principal=r.principal, version=r.version,
              messages=(('system','Reply briefly.'),('user',r.body)),
              template_revision=r.template_revision)

    def _cancel_due(self, sid:str, tick:int) -> bool:
        r=self.by_sid[sid]
        return r.cancel_after_ticks is not None and tick-r.arrival_tick>=r.cancel_after_ticks

    def _deadline_due(self,sid:str,tick:int) -> bool:
        return tick-self.by_sid[sid].arrival_tick>=self.by_sid[sid].patience_ticks

    def _finish(self,sid:str, reason:str, tick:int) -> None:
        require(sid not in self.terminal and sid in self.by_sid, '重复终止或未知sid')
        require(reason in {'completed_eos','completed_cap','queue_rejected','oversize_rejected',
                            'cancelled_waiting','cancelled_active','deadline_waiting',
                            'deadline_active','scheduler_timeout','kv_error'}, '终态无效')
        if sid in self.active:
            self.service.close(sid)
            session=self.active.pop(sid)
            decoded=session['decoded']
            first=session['first_token_tick']
        else:
            decoded=0;first=None
        row={'sid':sid,'principal':self.by_sid[sid].principal,'reason':reason,
             'arrival_tick':self.by_sid[sid].arrival_tick,'end_tick':tick,
             'started_tick':self.admitted[sid]['tick'] if sid in self.admitted else None,
             'first_output_tick':first,'decoded_tokens':decoded}
        self.terminal[sid]=row

    def _plan(self, sid:str):
        """只读前缀预检，不更新LRU；缓存损坏必须抛异常，不能转成miss。"""
        req=self._request(sid)
        ids=request_tokens(self.service.tok,req)
        identity=make_identity(req,self.service.authority,self.service.tok,
                              self.service.models,self.service.runner.model_hashes)
        keys=prefix_keys(identity,ids,self.cfg.block_size)
        found=[]
        for i,key in enumerate(keys):
            entry=self.pool.cache.get(key)
            if entry is None:break
            self.pool._validate_entry(entry,identity,tuple(ids[:(i+1)*self.cfg.block_size]),key)
            found.append(entry)
        block_count=math.ceil(len(ids)/self.cfg.block_size)
        max_blocks=math.ceil((len(ids)+self.by_sid[sid].decode_steps-1)/self.cfg.block_size)
        hits=len(found)
        safe_to_evict=sum(e.physical_page not in {f.physical_page for f in found}
                          and self.pool.refs[e.physical_page]==1 for e in self.pool.cache.values())
        # 活动请求未来可能追加token，预留账本并非真实分配；避免无意在decode时OOM。
        reserved=0
        for run_sid,info in self.active.items():
            s=self.pool.sessions[run_sid]
            remaining=self.by_sid[run_sid].decode_steps-info['decoded']
            reserved+=max(0,math.ceil((len(s.tokens)+remaining-1)/self.cfg.block_size)-len(s.block_table))
        available=max(0,len(self.pool.free)+safe_to_evict-reserved)
        return {'ids':ids,'identity':identity,'hit_blocks':hits,
                'hit_keys':[e.key for e in found], 'prompt_blocks':block_count,
                'required_new':max_blocks-hits,'available_new':available,
                'oversize':max_blocks>self.cfg.capacity_blocks}

    def _admit(self,sid,tick:int,plan:dict):
        r=self.by_sid[sid]
        admit=self.service.admit(self._request(sid))
        require(admit['hit_blocks']==plan['hit_blocks'] and
                admit['prompt_tokens']==len(plan['ids']),
                '预检与执行之间的缓存不一致；此单线程教学调度器不允许竞态')
        hit=admit['hit_tokens'];prompt=admit['prompt_tokens']
        self.total_prompt_tokens+=prompt
        self.total_eligible_tokens+=prompt//self.cfg.block_size*self.cfg.block_size
        self.prompts_reused+=hit
        self.prompts_computed+=admit['new_tokens_computed']
        self.hit_requests+=int(hit>0)
        self.admitted[sid]={'tick':tick,'hit_tokens':hit,'prompt_tokens':prompt,
                            'computed_tokens':admit['new_tokens_computed'],
                            'max_logit_delta':admit['max_logits_abs_delta']}
        self.active[sid]={'started_tick':tick,'decoded':0,'first_token_tick':None}
        self.rr_admit_order.append(sid)
        self.pending.remove(sid)
        self.peak_active=max(self.peak_active,len(self.active))

    def _available_tenant(self,principal:str)->bool:
        return sum(self.by_sid[sid].principal==principal for sid in self.active)<self.cfg.per_principal_active

    def _choose(self):
        require(self.pending, '没有排队请求')
        if self.cfg.policy=='fifo':return [self.pending[0]]
        result=[]
        for offset in range(len(self.rr_tenants)):
            tenant=self.rr_tenants[(self.rr_next+offset)%len(self.rr_tenants)]
            match=next((sid for sid in self.pending if self.by_sid[sid].principal==tenant),None)
            if match is not None:result.append(match)
        return result

    def _admit_waiting(self,tick:int):
        while self.pending and len(self.active)<self.cfg.max_active:
            selected=None;plan=None;handled_oversize=False
            for sid in self._choose():
                principal=self.by_sid[sid].principal
                if not self._available_tenant(principal):
                    self.quota_deferrals+=1
                    continue
                proposed=self._plan(sid)
                if proposed['oversize']:
                    self.pending.remove(sid)
                    self._finish(sid,'oversize_rejected',tick)
                    handled_oversize=True
                    break
                if proposed['required_new']>proposed['available_new']:
                    self.capacity_deferrals+=1
                    continue
                selected=sid;plan=proposed
                break
            if handled_oversize:continue
            if selected is None:break
            self._admit(selected,tick,plan)
            if self.cfg.policy=='tenant_rr':
                pos=self.rr_tenants.index(self.by_sid[selected].principal)
                self.rr_next=(pos+1)%len(self.rr_tenants)
        if self.pending and len(self.active)>=self.cfg.max_active:self.slot_deferrals+=1

    def _decode(self,tick:int):
        for sid in list(self.active):
            if self._cancel_due(sid,tick):
                self._finish(sid,'cancelled_active',tick)
                continue
            if self._deadline_due(sid,tick):
                self._finish(sid,'deadline_active',tick)
                continue
            state=self.active[sid]
            if tick<=state['started_tick']:continue
            logits=self.service.runner.last_logits[sid]
            require(logits.shape==(self.service.tok.vocab_size,), '缓存logits形状错误')
            token=int(logits.argmax())
            state['decoded']+=1
            if state['first_token_tick'] is None:state['first_token_tick']=tick
            if token==self.service.tok.eos_token_id:
                self._finish(sid,'completed_eos',tick)
            elif state['decoded']>=self.by_sid[sid].decode_steps:
                self._finish(sid,'completed_cap',tick)
            else:
                try:self.service.continue_token(sid,token)
                except NoKVSpace:
                    self.model_errors+=1
                    self._finish(sid,'kv_error',tick)

    def _step(self,tick:int):
        # 相同tick中，所有请求先到达；若队列已满则明确记录拒绝。
        while self.arrival_cursor<len(self.items) and self.items[self.arrival_cursor].arrival_tick==tick:
            req=self.items[self.arrival_cursor]
            self.arrival_cursor+=1
            if len(self.pending)>=self.cfg.queue_limit:
                self._finish(req.sid,'queue_rejected',tick)
            else:self.pending.append(req.sid)
        self.queue_peak=max(self.queue_peak,len(self.pending))
        for sid in list(self.pending):
            if self._cancel_due(sid,tick):
                self.pending.remove(sid);self._finish(sid,'cancelled_waiting',tick)
            elif self._deadline_due(sid,tick):
                self.pending.remove(sid);self._finish(sid,'deadline_waiting',tick)
        self._decode(tick)
        self._admit_waiting(tick)
        self.pool.audit()
        usage=self.pool.cache_usage()
        self.peak_live_blocks=max(self.peak_live_blocks,usage['live_blocks'])
        self.active_time_slots+=len(self.active)
        self.trace.append({'tick':tick,'arrived':self.arrival_cursor,
                           'queued':len(self.pending),'active':len(self.active),
                           'terminal':len(self.terminal),'physical_live_blocks':usage['live_blocks'],
                           'idle_cached_blocks':usage['idle_cached_blocks'],
                           'cache_pins':usage['cache_pins']})

    def run(self):
        require(not self.finalized, '调度器只能运行一次')
        for tick in range(self.cfg.max_ticks):
            self._step(tick)
            if len(self.terminal)==len(self.items):break
        else:
            tick=self.cfg.max_ticks
            for sid in list(self.pending):
                self.pending.remove(sid);self._finish(sid,'scheduler_timeout',tick)
            for sid in list(self.active):self._finish(sid,'scheduler_timeout',tick)
        require(len(self.terminal)==len(self.items), '到达请求必须逐条进入终态')
        require(self.arrival_cursor==len(self.items), '调度上限必须覆盖全部到达请求')
        require(not self.pending and not self.active and not self.pool.sessions,
                '终态必须释放所有KV会话')
        self.finalized=True
        report=self.report()
        # 刷新最终的cache所有权；不更改已经计算的容量轨迹。
        flush=self.service.final_flush()
        require(flush['live_blocks']==0 and self.pool.data.count_nonzero().item()==0,
                '必须在所有场景后回收并清零物理KV')
        report['final_flush']={'live_blocks':flush['live_blocks'],
                              'cache_pins':flush['cache_pins'],'all_bytes_zero':True}
        return report

    def report(self):
        require(self.finalized,'只有结束全部请求才能汇总统计')
        kinds=Counter(row['reason'] for row in self.terminal.values())
        good={'completed_eos','completed_cap'}
        admitted_count=len(self.admitted)
        completed=sum(v for k,v in kinds.items() if k in good)
        wait=[r['tick']-self.by_sid[sid].arrival_tick for sid,r in self.admitted.items()]
        first=[r['first_output_tick']-r['arrival_tick'] for r in self.terminal.values()
               if r['first_output_tick'] is not None]
        tenants=sorted({r.principal for r in self.items})
        bytenant={}
        for principal in tenants:
            arrivals=sum(x.principal==principal for x in self.items)
            accepted=sum(self.by_sid[x].principal==principal for x in self.admitted)
            outcomes=Counter(r['reason'] for r in self.terminal.values() if r['principal']==principal)
            done=sum(v for k,v in outcomes.items() if k in good)
            bytenant[principal]={'arrived':arrivals,'admitted':accepted,'completed':done,
                 'completion_fraction':done/arrivals,'reasons':dict(sorted(outcomes.items()))}
        prompt_block_queries=sum(rec['prompt_tokens']//self.cfg.block_size for rec in self.admitted.values())
        hit_blocks=self.prompts_reused//self.cfg.block_size
        summary={'requests':len(self.items),'admitted':admitted_count,'completed':completed,
                 'terminal_counts':dict(sorted(kinds.items())),
                 'request_cache_hit_fraction':metric_ratio(self.hit_requests,admitted_count),
                 'prompt_token_hit_fraction':metric_ratio(self.prompts_reused,self.total_prompt_tokens),
                 'eligible_full_block_hit_fraction':metric_ratio(self.prompts_reused,self.total_eligible_tokens),
                 'cache_block_query_hit_fraction':metric_ratio(hit_blocks,prompt_block_queries),
                 'prefill_tokens_total':self.total_prompt_tokens,
                 'prefill_tokens_computed':self.prompts_computed,
                 'prefill_tokens_reused':self.prompts_reused,
                 'admitted_prompt_full_block_queries':prompt_block_queries,
                 'hit_blocks':hit_blocks,
                 'decode_output_tokens':sum(row['decoded_tokens'] for row in self.terminal.values()),
                 'actually_computed_model_tokens':self.service.runner.tokens_processed,
                 'max_logit_abs_delta':max((r['max_logit_delta'] for r in self.admitted.values()),default=0.0),
                 'completion_per_logical_tick':completed/len(self.trace),
                 'logical_ticks':len(self.trace),
                 'queue_peak':self.queue_peak,'active_peak':self.peak_active,
                 'physical_live_blocks_peak':self.peak_live_blocks,
                 'physical_capacity_blocks':self.cfg.capacity_blocks,
                 'kv_pool_bytes_preallocated':self.pool.data.numel()*self.pool.data.element_size(),
                 'kv_bytes_per_block':self.pool.bytes_per_block,
                 'logical_wait_ticks_p50':nearest_rank(wait,.5),
                 'logical_wait_ticks_p95':nearest_rank(wait,.95),
                 'logical_time_to_first_output_ticks_p95':nearest_rank(first,.95),
                 'queue_rejections':kinds['queue_rejected'],
                 'capacity_deferral_events':self.capacity_deferrals,
                 'tenant_quota_deferral_events':self.quota_deferrals,
                 'active_slot_deferral_events':self.slot_deferrals,
                 'kv_errors':self.model_errors,
                 'pool_evictions':self.pool.evictions,
                 'skipped_promotions':self.pool.skipped_promotions,
                 'cache_pins_before_flush':len(self.pool.cache),
                 'fairness_jain_completion_fraction':jain_index([bytenant[t]['completion_fraction'] for t in tenants]),
                 'per_principal':bytenant,
                 'notes':['logical_ticks_not_wallclock','not_TTFT_TPOT_or_GPU_goodput',
                          'cache_hits_are_admitted_prompt_tokens_only',
                          'Jain_index_is_not_fairness_guarantee']}
        require(self.prompts_computed+self.prompts_reused==self.total_prompt_tokens,
                '实际prefill token账本不守恒')
        require(sum(kinds.values())==len(self.items), '终态统计不守恒')
        require(self.peak_active<=self.cfg.max_active and
                self.peak_live_blocks<=self.cfg.capacity_blocks,'超过活动/物理页上限')
        return {'config':asdict(self.cfg),'summary':summary,'admitted':
                  [{**{'sid':sid},**self.admitted[sid]} for sid in sorted(self.admitted)],
                'terminal':[self.terminal[sid] for sid in sorted(self.terminal)],
                'trace':self.trace,'admit_order':self.rr_admit_order}

## 9. 新增 SLO 契约与百分位数

一条请求带来：`arrival_tick`、`ttft_slo_ticks`、`e2e_slo_ticks`、`patience_ticks`（绝对到期由到达加上耐心预算决定）。本课规定：`1 ≤ TTFT预算 ≤ E2E预算 ≤ patience`。

全部请求终态都进入 SLO 计数；首 token 与完成时延只在有相应事件的请求上统计，避免 `None` 被错误转为0。`P95/P99` 使用 nearest-rank：`ceil(p*n)` 位，不做隐式插值。很小样本上的 P99 常常只是最大观测值。

在模型调度中，prefill 零模拟时间、不模拟网络、客户端与真实 wall clock。因此最终所有单位均叫 `logical_ticks`。

In [15]:
# 新的SLO契约：到达、TTFT、端到端与独立统计
@dataclass(frozen=True)
class SLORequest(TrafficRequest):
    ttft_slo_ticks: int = 3
    e2e_slo_ticks: int = 6

    def validate(self):
        super().validate()
        require(type(self.ttft_slo_ticks) is int and self.ttft_slo_ticks >= 1,
                'TTFT SLO必须是正整数tick')
        require(type(self.e2e_slo_ticks) is int and self.e2e_slo_ticks >= self.ttft_slo_ticks,
                '端到端预算必须不小于首token预算')
        require(self.patience_ticks >= self.e2e_slo_ticks,
                '硬请求到期不能早于承诺的端到端预算')


@dataclass(frozen=True)
class SLOConfig(LoadConfig):
    policy: str = 'fifo'
    shed_unachievable: bool = False

    def validate(self):
        # 保留上一课物理容量、队列、租户配额的所有验证，扩展策略名。
        require(self.policy in {'fifo','tenant_rr','edf','fair_edf'}, '未知SLO调度策略')
        require(type(self.shed_unachievable) is bool, 'shed需要布尔值')
        LoadConfig(capacity_blocks=self.capacity_blocks,cache_limit=self.cache_limit,
                   block_size=self.block_size,max_active=self.max_active,
                   per_principal_active=self.per_principal_active,
                   queue_limit=self.queue_limit,max_ticks=self.max_ticks,
                   policy='fifo').validate()


def slo_passes(row: dict, request: SLORequest) -> bool:
    return (row['reason'] in {'completed_eos','completed_cap'} and
            row['first_output_tick'] is not None and
            row['first_output_tick']-request.arrival_tick <= request.ttft_slo_ticks and
            row['end_tick']-request.arrival_tick <= request.e2e_slo_ticks)


def honest_rate(numerator: int, denominator: int) -> float | None:
    """全到达请求是SLO准入/成功率的固定分母；空流量不伪装成100%。"""
    return metric_ratio(numerator,denominator)


def latency_samples(terminal: list[dict],by_sid: dict[str,SLORequest]) -> dict:
    """只对真正完成的响应定义E2E；TTFT统计实际发生首输出的响应。

    基础模型每活跃tick至多流出一token，decode全程每tick前进一次；
    因此连续输出间隔恰好1 tick。这只是离散模型的性质，非GPU ITL实测。
    """
    completed=[r for r in terminal if r['reason'] in {'completed_eos','completed_cap'}]
    first=[r for r in terminal if r['first_output_tick'] is not None]
    queue_wait=[r['started_tick']-r['arrival_tick'] for r in terminal
                if r['started_tick'] is not None]
    ttft=[r['first_output_tick']-r['arrival_tick'] for r in first]
    e2e=[r['end_tick']-r['arrival_tick'] for r in completed]
    # TTFT包括入队、调度tick；prefill在admit同tick完成，不消耗模拟时间。
    require(all(x>=0 for x in ttft+e2e+queue_wait),'负时延表示调度/记录错误')
    gaps=[];per_request_tpot=[]
    for r in completed:
        n=r['decoded_tokens']
        require(n>=1 and r['first_output_tick'] is not None,'完成响应必须至少流出一个token')
        require(r['end_tick']==r['first_output_tick']+n-1,
                '本课每tick连续decode的记录与终态不一致')
        if n>1:
            intervals=[1]*(n-1)
            gaps.extend(intervals)
            per_request_tpot.append((r['end_tick']-r['first_output_tick'])/(n-1))
    return {'queue_wait_ticks':queue_wait,'ttft_ticks':ttft,'e2e_ticks':e2e,
            'inter_token_gaps_ticks':gaps,'per_request_tpot_ticks':per_request_tpot,
            'queue_wait_p50':nearest_rank(queue_wait,.5),
            'queue_wait_p95':nearest_rank(queue_wait,.95),
            'ttft_p50':nearest_rank(ttft,.5),'ttft_p95':nearest_rank(ttft,.95),
            'ttft_p99':nearest_rank(ttft,.99),
            'e2e_p50':nearest_rank(e2e,.5),'e2e_p95':nearest_rank(e2e,.95),
            'e2e_p99':nearest_rank(e2e,.99),
            'gap_count':len(gaps),'tpot_request_count':len(per_request_tpot),
            'definitions':{'ttft':'first_output_tick - arrival_tick; only requests with first output',
             'e2e':'end_tick - arrival_tick; completed only',
             'tpot':'(last_output_tick-first_output_tick)/(decoded_tokens-1), completed n>=2',
             'itl':'successive output events, pooled; completed only',
             'queue':'admission_tick-arrival_tick; admitted only',
             'units':'abstract logical ticks, NOT milliseconds, not wallclock GPU metrics'}}

In [16]:
example=SLORequest('s1',0,'alice','Hello world!',patience_ticks=8,
                   ttft_slo_ticks=2,e2e_slo_ticks=5)
example.validate()
for first,end in [(2,5),(3,5),(2,6)]:
    record={'reason':'completed_cap','arrival_tick':0,'first_output_tick':first,'end_tick':end}
    print('first=',first,'end=',end,'满足全部SLO:',slo_passes(record,example))

first= 2 end= 5 满足全部SLO: True
first= 3 end= 5 满足全部SLO: False
first= 2 end= 6 满足全部SLO: False


## 10. FIFO、租户轮转、EDF、租户公平的 EDF

在相同物理容量与相同到达序列上比较：

| 策略 | 在等待队列中如何选择 |
|---|---|
| FIFO | 队头；该请求若受租户配额阻塞，可能形成队头阻塞 |
| tenant_rr | 按租户轮转，允许跳过当前不可调度租户 |
| EDF | 选择最近 TTFT 截止时刻，之后以 E2E 截止和到达顺序打破平局 |
| fair_edf | 租户间轮转，每个租户内部优先最近截止的请求 |

**EDF 只对等待队列排序，不抢占已经运行的 decode。** `fair_edf` 也不是严格加权公平调度器：只作为一个可审计的小型策略对照。真实 Triton 支持优先队列与最大等待/超时策略，但具体调度语义和本课不完全相同。[NVIDIA Triton Batcher](https://docs.nvidia.com/deeplearning/triton-inference-server/archives/triton-inference-server-2580/user-guide/docs/user_guide/batcher.html)

In [17]:
# SLO调度器：FIFO、租户轮转、EDF与租户公平的EDF
class SLOScheduler(LoadScheduler):
    """真实CPU微型Transformer+真实物理KV；仅调度决策基于抽象tick。

    `edf`只对WAITING请求排序：运行中的decode不会被抢占。
    `fair_edf`按主体RR，主体内最早TTFT截止；不是严格公平保证。
    `shed_unachievable`只在不可能满足TTFT预算时丢弃尚未入场的请求。
    """
    def __init__(self,service:PrefixCacheRunner,config:SLOConfig,items:list[SLORequest]):
        require(all(isinstance(r,SLORequest) for r in items), '本课只接受带SLO的请求')
        for req in items:req.validate()
        super().__init__(service,config,items)
        self.slo_shed_count=0
        self.shed_events:list[dict]=[]

    def _choose(self):
        require(self.pending,'队列为空')
        if self.cfg.policy=='edf':
            return sorted(self.pending,key=lambda sid:(
                self.by_sid[sid].arrival_tick+self.by_sid[sid].ttft_slo_ticks,
                self.by_sid[sid].arrival_tick+self.by_sid[sid].e2e_slo_ticks,
                self.by_sid[sid].arrival_tick,sid))
        if self.cfg.policy=='fair_edf':
            selected=[]
            for offset in range(len(self.rr_tenants)):
                tenant=self.rr_tenants[(self.rr_next+offset)%len(self.rr_tenants)]
                available=[sid for sid in self.pending if self.by_sid[sid].principal==tenant]
                if available:
                    selected.append(min(available,key=lambda sid:(
                        self.by_sid[sid].arrival_tick+self.by_sid[sid].ttft_slo_ticks,
                        self.by_sid[sid].arrival_tick+self.by_sid[sid].e2e_slo_ticks,
                        self.by_sid[sid].arrival_tick,sid)))
            return selected
        return super()._choose()

    def _admit(self,sid,tick:int,plan:dict):
        super()._admit(sid,tick,plan)
        if self.cfg.policy=='fair_edf':
            self.rr_next=(self.rr_tenants.index(self.by_sid[sid].principal)+1)%len(self.rr_tenants)

    def _finish(self,sid:str,reason:str,tick:int):
        if reason!='slo_unachievable_rejected':
            return super()._finish(sid,reason,tick)
        require(sid not in self.active and sid in self.by_sid and sid not in self.terminal,
                'SLO舍弃仅允许尚未入场的合法请求')
        req=self.by_sid[sid]
        self.terminal[sid]={'sid':sid,'principal':req.principal,'reason':reason,
            'arrival_tick':req.arrival_tick,'end_tick':tick,'started_tick':None,
            'first_output_tick':None,'decoded_tokens':0}

    def _admit_waiting(self,tick:int):
        if self.cfg.shed_unachievable:
            for sid in list(self.pending):
                req=self.by_sid[sid]
                # 入场当tick做prefill，首个decode token最早是下一tick。
                if tick+1-req.arrival_tick>req.ttft_slo_ticks:
                    self.pending.remove(sid)
                    self._finish(sid,'slo_unachievable_rejected',tick)
                    self.slo_shed_count+=1
                    self.shed_events.append({'sid':sid,'decision_tick':tick,
                        'earliest_first_output_tick':tick+1,
                        'absolute_ttft_deadline':req.arrival_tick+req.ttft_slo_ticks})
        super()._admit_waiting(tick)

    def report(self):
        report=super().report()
        terminal=report['terminal']
        byid=self.by_sid
        success={r['sid'] for r in terminal if slo_passes(r,byid[r['sid']])}
        completed={r['sid'] for r in terminal if r['reason'] in {'completed_eos','completed_cap'}}
        counts=Counter(r['reason'] for r in terminal)
        total=len(self.items)
        tenant_summary={}
        for tenant in sorted({r.principal for r in self.items}):
            arrived=[r for r in self.items if r.principal==tenant]
            admitted=sum(r['principal']==tenant for r in report['terminal'] if r['started_tick'] is not None)
            timely=sum(r.sid in success for r in arrived)
            tenant_summary[tenant]={'arrived':len(arrived),'admitted':admitted,
                'slo_good':timely,'slo_attainment_all_arrivals':honest_rate(timely,len(arrived)),
                'completion_fraction_all_arrivals':honest_rate(
                    sum(r.sid in completed for r in arrived),len(arrived))}
        latency=latency_samples(terminal,byid)
        summary=report['summary']
        summary.update({'timely_completed':len(success),
          'slo_goodput_per_logical_tick':len(success)/len(report['trace']),
          'slo_attainment_all_arrivals':honest_rate(len(success),total),
          'slo_attainment_completed_only':honest_rate(len(success),len(completed)),
          'slo_shed_rejections':self.slo_shed_count,
          'all_arrival_rejection_fraction':honest_rate(counts['queue_rejected']+
             counts['oversize_rejected']+counts['slo_unachievable_rejected'],total),
          'slo_by_tenant':tenant_summary,
          'jain_slo_attainment':jain_index([x['slo_attainment_all_arrivals'] for x in tenant_summary.values()]),
          'latency':latency,
          'notes_slo':[
              'logical_ticks_not_milliseconds',
              'prefill_done_at_admission_tick_not_time_modeled',
              'ttft_stats_exclude_requests_without_first_token',
              'e2e_percentiles_include_completed_only_and_are_selection_biased',
              'all_arrivals_denominator_includes_shed_and_queue_reject',
              'edf_waiting_only_no_preemption',
              'jain_index_not_service_fairness_guarantee',
              'P99_with_fewer_than_100_values_is_sample_max_under_nearest_rank']})
        report['slo_contract']={r.sid:{'ttft_slo_ticks':r.ttft_slo_ticks,
            'e2e_slo_ticks':r.e2e_slo_ticks,'patience_ticks':r.patience_ticks} for r in self.items}
        report['slo_decisions']=sorted(self.shed_events,key=lambda x:x['sid'])
        require(sum(counts.values())==total and len(self.terminal)==total,'请求终态不守恒')
        require(len(success)<=len(completed)<=len(self.admitted)<=total,'及时完成统计不守恒')
        require(summary['slo_shed_rejections']==counts['slo_unachievable_rejected'],
                'SLO拒绝计数不守恒')
        return report

## 11. SLO 早拒：有些排队工作已经不可能赶上首 token 截止

在本课离散执行规则中，时刻 `t` 准入的请求，最快 `t+1` 得到第一 token。因此如果：

\[
(t+1)-arrival > TTFT\_budget
\]

这条**仍在等待队列**中的请求已不可能满足首 token 预算。`shed_unachievable=True` 时可在没有分配 KV 的情况下给出明确终态 `slo_unachievable_rejected`，不吞掉或重试请求。

边界必须使用 `>`，不是 `>=`：恰好满足截止允许入场。这个规则是本课模型下的**充分不可能条件**，不等价于真实GPU执行时间预测，更不是“预测所有提前拒绝都提高总体体验”。

所有拒绝继续计入全到达分母。条件完成成功率可能因“只保留容易完成者”而被人为抬高，所以必须同时报告全到达成功数、拒绝数与公平性。

In [18]:
print('到达t=0, 预算2时：')
for tick in (0,1,2,3):
    print('当前t=',tick,'最早首输出=',tick+1,
          '仍可赶上:',tick+1<=2)

到达t=0, 预算2时：
当前t= 0 最早首输出= 1 仍可赶上: True
当前t= 1 最早首输出= 2 仍可赶上: True
当前t= 2 最早首输出= 3 仍可赶上: False
当前t= 3 最早首输出= 4 仍可赶上: False


## 12. 可重复到达序列、相同预算对照与真实KV核算

三个负载形态：稳定流量（12请求）、偏斜突发（25请求）、混合截止（21请求）。每个组合按固定顺序构造，不使用真实用户内容或外部缓存。

**固定八组场景：**

1. `steady_fifo`：作为无压力对照。
2. `overload_fifo`、`overload_tenant_rr`、`overload_edf`、`overload_fair_edf`：同一个过载到达序列。
3. `overload_fair_edf_shed`：上一组相同请求上启用早拒。
4. `mixed_fair_edf` 与 `mixed_fair_edf_shed`：同一混合截止到达序列。

每个场景重新构建空 KV 池，模型参数完全一样。缓存会随场景内部请求自然预热，但不跨场景保留物理页；这避免了把先后实验的缓存状态当作政策优势。

可比较及时完成数、全到达比例、完成者条件比例、租户各自成功率、尾延迟条件分位数、KV 真正计算量、容量占用与全部拒绝原因。

In [19]:
# 可重复的请求到达与对照情景

def make_slo_workload(kind:str,count:int=24) -> list[SLORequest]:
    require(kind in {'steady','overload','mixed_deadlines','hot'},'未知SLO工作负载')
    require(type(count) is int and 4<=count<=80,'请求数4..80')
    items=[]
    for i in range(count):
        if kind=='steady':
            arrival=i*3
            tenant='alice' if i%2 else 'bob'
            body='Hello world! '+('repeat' if i%3 else 'alternate')
            ttft,e2e=3,6
        elif kind=='overload':
            # 两租户不均匀到达，故意制造队头阻塞与尾延迟压力。
            arrival=(i//5)*2
            tenant='bob' if i%5==0 else 'alice'
            body='Hello world! '+('repeat' if i%3 else f'cold{i:02d}')
            ttft,e2e=(3,7) if tenant=='bob' else (2,5)
        elif kind=='mixed_deadlines':
            arrival=i//3
            tenant='bob' if i%4==0 else 'alice'
            body='Hello world! '+('repeat' if i%2 else f'variant{i:02d}')
            ttft,e2e=(2,5) if i%3==0 else (5,9)
        else:
            arrival=i*2
            tenant='alice' if i%3 else 'alice_worker'
            body='Hello world! repeat'
            ttft,e2e=2,5
        items.append(SLORequest(sid=f'{kind[:3]}_{i:03d}',arrival_tick=arrival,
            principal=tenant,body=body,patience_ticks=e2e+3,
            decode_steps=2,ttft_slo_ticks=ttft,e2e_slo_ticks=e2e))
    validate_workload(items)
    return items


def new_environment(tok,models,config:SLOConfig):
    config.validate()
    authority=TrustedScopeAuthority({'alice':'private_alice',
        'alice_worker':'private_alice','bob':'private_bob',
        'org1':'trusted_org','org2':'trusted_org'})
    pool=PrefixKVPool(config.capacity_blocks,config.block_size,
        next(iter(models.values())).cfg,set(models),cache_capacity_blocks=config.cache_limit)
    return PrefixCacheRunner(tok,models,pool,authority)


def run_scenario(tok,models,config:SLOConfig,items:list[SLORequest]) -> dict:
    torch.set_num_threads(1)
    service=new_environment(tok,models,config)
    result=SLOScheduler(service,config,items).run()
    require(result['final_flush']['all_bytes_zero'] and result['final_flush']['live_blocks']==0,
            '终态必须清零真实KV页面')
    return result


def scenario_specs():
    overload=make_slo_workload('overload',25)
    steady=make_slo_workload('steady',12)
    mixed=make_slo_workload('mixed_deadlines',21)
    base=dict(capacity_blocks=36,cache_limit=18,max_active=2,
              per_principal_active=1,queue_limit=12,max_ticks=85)
    return [
      ('steady_fifo',SLOConfig(**base,policy='fifo'),steady),
      ('overload_fifo',SLOConfig(**base,policy='fifo'),overload),
      ('overload_tenant_rr',SLOConfig(**base,policy='tenant_rr'),overload),
      ('overload_edf',SLOConfig(**base,policy='edf'),overload),
      ('overload_fair_edf',SLOConfig(**base,policy='fair_edf'),overload),
      ('overload_fair_edf_shed',SLOConfig(**base,policy='fair_edf',shed_unachievable=True),overload),
      ('mixed_fair_edf',SLOConfig(**base,policy='fair_edf'),mixed),
      ('mixed_fair_edf_shed',SLOConfig(**base,policy='fair_edf',shed_unachievable=True),mixed),
    ]


def scenario_summary(cases:dict[str,dict]) -> dict:
    require(all(name in cases for name in ('overload_fifo','overload_tenant_rr',
            'overload_edf','overload_fair_edf','overload_fair_edf_shed')),
            '缺失同一过载工作负载的调度策略对照')
    table=[]
    for name,case in cases.items():
        s=case['summary']
        table.append({'scenario':name,'arrived':s['requests'],
          'completed':s['completed'],'slo_good':s['timely_completed'],
          'slo_attainment_all_arrivals':s['slo_attainment_all_arrivals'],
          'slo_attainment_completed_only':s['slo_attainment_completed_only'],
          'queue_rejected':s['queue_rejections'],'slo_shed':s['slo_shed_rejections'],
          'first_token_p95_ticks':s['latency']['ttft_p95'],
          'end_to_end_p99_ticks':s['latency']['e2e_p99'],
          'slo_goodput_per_tick':s['slo_goodput_per_logical_tick'],
          'jain_slo_attainment':s['jain_slo_attainment'],
          'prefill_tokens_computed':s['prefill_tokens_computed'],
          'prefill_tokens_reused':s['prefill_tokens_reused'],
          'physical_live_blocks_peak':s['physical_live_blocks_peak']})
    overload_source=cases['overload_fifo']['slo_contract']
    require(all(cases[name]['slo_contract']==overload_source for name in
        ('overload_tenant_rr','overload_edf','overload_fair_edf','overload_fair_edf_shed')),
        '策略对照必须使用相同的到达、SLO契约和请求身份')
    require(all(v['final_flush']['all_bytes_zero'] for v in cases.values()),'场景未清零真实KV')
    return {'table':table,'definitions':{
        'goodput':'slo_good_count / logical_ticks_including_arrivals; not GPU tokens per second',
        'attainment':'slo_good_count / all_arrived_requests including rejected/cancelled/timeouts',
        'latency':'nearest-rank conditional on output/completion; abstract ticks not milliseconds'},
        'limitations':['toy CPU model with real KV but simulated scheduling ticks',
                       'no actual GPU wall-clock TTFT/TPOT',
                       'no statistical inference from tiny P99 sample',
                       'shedding may raise conditional percentage while hurting all-arrival outcome']}

In [20]:
for name,cfg,items in scenario_specs():
    print(f'{name:27s}',len(items),'个请求',
          '策略=',cfg.policy,'早拒=',cfg.shed_unachievable,
          '队列上限=',cfg.queue_limit,'物理页上限=',cfg.capacity_blocks)

steady_fifo                 12 个请求 策略= fifo 早拒= False 队列上限= 12 物理页上限= 36
overload_fifo               25 个请求 策略= fifo 早拒= False 队列上限= 12 物理页上限= 36
overload_tenant_rr          25 个请求 策略= tenant_rr 早拒= False 队列上限= 12 物理页上限= 36
overload_edf                25 个请求 策略= edf 早拒= False 队列上限= 12 物理页上限= 36
overload_fair_edf           25 个请求 策略= fair_edf 早拒= False 队列上限= 12 物理页上限= 36
overload_fair_edf_shed      25 个请求 策略= fair_edf 早拒= True 队列上限= 12 物理页上限= 36
mixed_fair_edf              21 个请求 策略= fair_edf 早拒= False 队列上限= 12 物理页上限= 36
mixed_fair_edf_shed         21 个请求 策略= fair_edf 早拒= True 队列上限= 12 物理页上限= 36


## 13. 完整运行：真正执行全部物理KV + SLO场景

下面的完整演练默认开启，不需要此前课程输出文件。报告里包含：每个请求的唯一终态、排队/TTFT/E2E 逻辑 tick、SLO、租户成功率、真实KV块峰值、真实增量 logits 与完整前向差异、拒绝依据、逐 tick 资源轨迹。

所有原始输入文本只存在于本次构造的教学负载，不写入终态表；具体真实系统还需做访问控制和隐私审计，本课不声称是生产安全实现。

In [21]:
# 独立权重恢复与可复跑实验报告
def save_models(path:Path,tok,models,seed:int):
    blob={'version':VERSION,'seed':seed,
      'model_config':asdict(next(iter(models.values())).cfg),
      'tokenizer':tok.contract(),
      'hashes':{key:state_hash(m.state_dict()) for key,m in models.items()},
      'states':{key:{k:t.detach().cpu().clone() for k,t in m.state_dict().items()}
                for key,m in models.items()}}
    torch.save(blob,path)


def load_models(path:Path):
    blob=torch.load(path,map_location='cpu',weights_only=True)
    require(blob['version']==VERSION,'不是第76课受信模型版本')
    tok,models=build_models(blob['seed'])
    require(blob['model_config']==asdict(next(iter(models.values())).cfg) and
            blob['tokenizer']==tok.contract(),'模型或分词契约不匹配')
    require(set(models)==set(blob['states'])==set(blob['hashes']),'版本集合不匹配')
    for name,m in models.items():
        require(state_hash(blob['states'][name])==blob['hashes'][name],
                '模型权重字节摘要不匹配')
        m.load_state_dict(blob['states'][name],strict=True)
        require(state_hash(m.state_dict())==blob['hashes'][name],'加载后参数不一致')
        m.eval().requires_grad_(False)
    return tok,models,blob['seed']


def reload_run(folder:Path)->dict:
    folder=Path(folder)
    tok,models,seed=load_models(folder/'models.pt')
    expected=json.loads((folder/'overload_fair_edf_shed.json').read_text(encoding='utf-8'))
    cfg,items=next((cfg,items) for n,cfg,items in scenario_specs() if n=='overload_fair_edf_shed')
    report=run_scenario(tok,models,cfg,items)
    require(report==expected,'干净对象重放与保存的SLO/KV报告不一致')
    return {'model_state_hash_checked':True,'scenario':'overload_fair_edf_shed',
            'exact_report_equal':True,'all_physical_kv_freed':report['final_flush']['all_bytes_zero'],
            'not_gpu_latency_or_live_service_checkpoint':True}

In [22]:
def fixture_demo(output_root:Path=Path('lesson_76_outputs'),seed:int=7601)->Path:
    torch.set_num_threads(1)
    seed_all(seed)
    root=Path(output_root)
    path=root/datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%fZ')
    path.mkdir(parents=True,exist_ok=False)
    manifest={'version':VERSION,'seed':seed,'python':platform.python_version(),
              'torch':str(torch.__version__),'device':'cpu','status':'running',
              'sources':SOURCES,'scope':'real CPU KV+logical scheduler, not GPU ms metrics',
              'admission':'same-tick prefill has zero simulated time; first output earliest next tick'}
    write_json(path/'manifest.json',manifest)
    try:
        tok,models=build_models(seed)
        save_models(path/'models.pt',tok,models,seed)
        cases={}
        for name,cfg,items in scenario_specs():
            report=run_scenario(tok,models,cfg,items)
            cases[name]=report
            write_json(path/(name+'.json'),report)
            print(name,'SLO及时完成',report['summary']['timely_completed'],'/',len(items),
                  '拒绝',report['summary']['queue_rejections']+report['summary']['slo_shed_rejections'])
        write_json(path/'summary.json',scenario_summary(cases))
        reload_report=reload_run(path)
        write_json(path/'reload.json',reload_report)
        manifest['status']='completed';write_json(path/'manifest.json',manifest)
        print('结果目录:',path)
        return path
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        write_json(path/'manifest.json',manifest)
        raise

In [23]:
RUN_FULL_DEMO = True
run_folder = None
if RUN_FULL_DEMO:
    run_folder=fixture_demo(Path('lesson_76_outputs'),seed=7601)
else:
    print('已跳过完整演练，没有预填或伪造数据。')

steady_fifo SLO及时完成 12 / 12 拒绝 0


overload_fifo SLO及时完成 2 / 25 拒绝 7


overload_tenant_rr SLO及时完成 6 / 25 拒绝 5


overload_edf SLO及时完成 6 / 25 拒绝 5


overload_fair_edf SLO及时完成 6 / 25 拒绝 5
overload_fair_edf_shed SLO及时完成 10 / 25 拒绝 15


mixed_fair_edf SLO及时完成 7 / 21 拒绝 3
mixed_fair_edf_shed SLO及时完成 11 / 21 拒绝 10


结果目录: lesson_76_outputs/run_20261105_011018_441224Z


## 14. 对照结果：绝对完成数与条件比例一起读

下面会从实际生成的 `summary.json` 读取八组结果，而不是写死一个“预期改进”数字。默认 CPU 实验中，过载 FIFO 及时完成 **2/25**，租户轮转 **6/25**，`fair_edf + shed` **10/25**，同时出现了 **15 次准入舍弃**。

对于后者，已完成请求里的条件成功率为 **10/10=100%**，但**全到达成功率仍只有 10/25=40%**。只展示100%会误导。

这不证明早拒普遍优越：多租户奖励、延迟、丢弃成本和业务优先级都应独立评估。Jain 指数这里只用于比较各租户的**及时完成占到达比例**，不能代替真实租户公平或配额保证。

In [24]:
if run_folder is not None:
    report=json.loads((run_folder/'summary.json').read_text(encoding='utf-8'))
    print('场景                         及时完成  全到达SLO  完成者条件SLO  早拒  P95_TTFT_tick P99_E2E_tick')
    for item in report['table']:
        print(f"{item['scenario']:28s} {item['slo_good']:3d}/{item['arrived']:<3d}    "
              f"{item['slo_attainment_all_arrivals']:.3f}      "
              f"{item['slo_attainment_completed_only']!s:>5}         "
              f"{item['slo_shed']:3d}       "
              f"{str(item['first_token_p95_ticks']):>3s}            "
              f"{str(item['end_to_end_p99_ticks']):>3s}")

场景                         及时完成  全到达SLO  完成者条件SLO  早拒  P95_TTFT_tick P99_E2E_tick
steady_fifo                   12/12     1.000        1.0           0         1              2
overload_fifo                  2/25     0.080       0.25           0         7              8
overload_tenant_rr             6/25     0.240       0.75           0         7              6
overload_edf                   6/25     0.240       0.75           0         7              6
overload_fair_edf              6/25     0.240       0.75           0         7              6
overload_fair_edf_shed        10/25     0.400        1.0          15         1              2
mixed_fair_edf                 7/21     0.333      0.6363636363636364           0        10             11
mixed_fair_edf_shed           11/21     0.524        1.0           9         5              6


In [25]:
if run_folder is not None:
    regular=json.loads((run_folder/'overload_fair_edf.json').read_text(encoding='utf-8'))['summary']
    shed=json.loads((run_folder/'overload_fair_edf_shed.json').read_text(encoding='utf-8'))['summary']
    for label,s in [('不早拒',regular),('早拒',shed)]:
        print(label,' | 到达/已完成/及时:',s['requests'],s['completed'],s['timely_completed'],
              ' | 队列拒绝/主动早拒:',s['queue_rejections'],s['slo_shed_rejections'],
              ' | 租户结果:',s['slo_by_tenant'],
              ' | 本场景P99样本数:',len(s['latency']['e2e_ticks']))

不早拒  | 到达/已完成/及时: 25 8 6  | 队列拒绝/主动早拒: 5 0  | 租户结果: {'alice': {'admitted': 8, 'arrived': 20, 'completion_fraction_all_arrivals': 0.15, 'slo_attainment_all_arrivals': 0.05, 'slo_good': 1}, 'bob': {'admitted': 5, 'arrived': 5, 'completion_fraction_all_arrivals': 1.0, 'slo_attainment_all_arrivals': 1.0, 'slo_good': 5}}  | 本场景P99样本数: 8
早拒  | 到达/已完成/及时: 25 10 10  | 队列拒绝/主动早拒: 0 15  | 租户结果: {'alice': {'admitted': 5, 'arrived': 20, 'completion_fraction_all_arrivals': 0.25, 'slo_attainment_all_arrivals': 0.25, 'slo_good': 5}, 'bob': {'admitted': 5, 'arrived': 5, 'completion_fraction_all_arrivals': 1.0, 'slo_attainment_all_arrivals': 1.0, 'slo_good': 5}}  | 本场景P99样本数: 10


## 15. 资源守恒与重放检查

物理页硬上限在每 tick 约束真实 KV 张量；结束后所有会话、缓存所有权与非零 KV 字节归零。模型、LoRA、tokenizer 模板身份在保存时绑定，并且从新实例独立恢复。

报告重放的是确定性场景 `overload_fair_edf_shed`，不是把某次在线服务的内存堆与活跃 KV 会话冻结后再恢复，也没有模拟网络或 GPU 设备漂移。只加载可信本地 checkpoint。

In [26]:
if run_folder is not None:
    print(json.dumps(reload_run(run_folder),ensure_ascii=False,indent=2))

{
  "model_state_hash_checked": true,
  "scenario": "overload_fair_edf_shed",
  "exact_report_equal": true,
  "all_physical_kv_freed": true,
  "not_gpu_latency_or_live_service_checkpoint": true
}


## 16. 自动检查：不能以一次顺利输出代替边界证明

覆盖：百分位 nearest-rank、空分母、端到端/首token 等号、队头阻塞、EDF 选择、租户轮转、早拒必须无 KV、缓存守恒、真实 logits 误差、唯一终态与物理页归零、两种SLO统计分母、租户公平和模型权重只读。

检验是离线 CPU 教学模型的契约，不是现实服务性能/安全审计。任何“逻辑 tick 数”都不要改写成“毫秒”或拿来承诺 P99 时延。

In [27]:
# 可执行检查：数学、SLO窗口、真实物理KV与负载机制
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,ok):
        if not bool(ok):raise AssertionError(name)
        self.names.append(name)
    def close(self,name,a,b,atol=1e-8):
        self.ok(name,a is not None and b is not None and abs(float(a)-float(b))<=atol)
    def rejects(self,name,fn):
        try:fn()
        except (LessonError,ValueError):self.names.append(name);return
        raise AssertionError(name+': 应显式拒绝而不是继续执行')


def run_checks() -> dict:
    torch.set_num_threads(1)
    seed_all(7601)
    c=Checks()
    c.ok('SLO版本区别于上一课',VERSION=='lesson76-slo-admission-v1')
    c.close('空时延窗口不造出0或100',int(honest_rate(0,0) is None),1)
    c.close('总体到达比例',honest_rate(3,8),.375)
    c.rejects('非法成功数拒绝',lambda:honest_rate(9,8))
    c.close('P50 nearest-rank',nearest_rank([2,8,1,6],.5),2)
    c.close('P95在20以下几乎是最大值',nearest_rank([1,3,6,8,9],.95),9)
    c.close('P99在小样本退化为最大值',nearest_rank([1,3,6,8,9],.99),9)
    c.ok('空P99无定义',nearest_rank([],.99) is None)
    c.rejects('分位数大于1拒绝',lambda:nearest_rank([1,2],1.01))
    c.rejects('负时间拒绝',lambda:nearest_rank([-1,1],.5))
    base=SLORequest('r1',0,'alice','Hello world!',patience_ticks=8,
                    ttft_slo_ticks=2,e2e_slo_ticks=5)
    base.validate()
    row={'reason':'completed_cap','first_output_tick':2,'end_tick':5,'arrival_tick':0}
    c.ok('正好满足TTFT及E2E边界',slo_passes(row,base))
    c.ok('超过TTFT拒绝及时完成',not slo_passes({**row,'first_output_tick':3},base))
    c.ok('超过E2E拒绝及时完成',not slo_passes({**row,'end_tick':6},base))
    c.ok('已被queue拒绝不计SLO成功',not slo_passes({**row,'reason':'queue_rejected'},base))
    c.ok('无首token不计成功',not slo_passes({**row,'first_output_tick':None},base))
    c.rejects('零TTFT预算拒绝',lambda:SLORequest('a',0,'alice','x',ttft_slo_ticks=0).validate())
    c.rejects('E2E小于TTFT拒绝',lambda:SLORequest('a',0,'alice','x',ttft_slo_ticks=6,e2e_slo_ticks=5).validate())
    c.rejects('硬TTL早于SLO拒绝',lambda:SLORequest('a',0,'alice','x',patience_ticks=4,e2e_slo_ticks=5).validate())
    c.rejects('非法租户身份拒绝',lambda:SLORequest('a',0,'mallory','x').validate())
    c.rejects('未知策略拒绝',lambda:SLOConfig(policy='priority_magic').validate())
    c.rejects('缓存物理上限非法拒绝',lambda:SLOConfig(capacity_blocks=10,cache_limit=11).validate())
    c.rejects('单租户并发不能超过全局',lambda:SLOConfig(max_active=1,per_principal_active=2).validate())
    c.rejects('布尔shed不能用整数',lambda:SLOConfig(shed_unachievable=1).validate())
    sample=[{'reason':'completed_cap','arrival_tick':0,'started_tick':1,
             'first_output_tick':2,'end_tick':3,'decoded_tokens':2,'sid':'x'},
            {'reason':'queue_rejected','arrival_tick':0,'started_tick':None,
             'first_output_tick':None,'end_tick':0,'decoded_tokens':0,'sid':'y'},
            {'reason':'deadline_active','arrival_tick':0,'started_tick':1,
             'first_output_tick':2,'end_tick':6,'decoded_tokens':1,'sid':'z'}]
    measured=latency_samples(sample,{'x':base,'y':base,'z':base})
    c.ok('端到端分位数只计算完成请求',measured['e2e_ticks']==[3])
    c.ok('TTFT包含有首token但后续失败的请求',measured['ttft_ticks']==[2,2])
    c.ok('排队统计只看已准入',measured['queue_wait_ticks']==[1,1])
    c.ok('ITL只按相邻输出事件记录',measured['inter_token_gaps_ticks']==[1])
    c.ok('单token完成没有TPOT样本',measured['tpot_request_count']==1)
    c.close('样本TTFT P95',measured['ttft_p95'],2)
    c.close('样本E2E P99',measured['e2e_p99'],3)
    c.rejects('不连续decode时间戳拒绝',lambda:latency_samples([{**sample[0],'end_tick':4}],{'x':base}))
    c.rejects('完成无输出拒绝',lambda:latency_samples([{**sample[0],'first_output_tick':None}],{'x':base}))
    c.ok('每请求TPOT与全体ITL取均值可能不同',
         (1+5)/2!=(1+5+5+5)/4)
    # 本课使用真实多版本模型、真实物理KV页、可信身份；SLO tick是额外模拟层。
    tok,models=build_models(7601)
    prior_hash={k:state_hash(m.state_dict()) for k,m in models.items()}
    c.ok('两个冻结LoRA版本可使用',set(models)=={'v0001','v0002'})
    c.ok('所有模型参数被冻结',all(not p.requires_grad for m in models.values() for p in m.parameters()))
    c.ok('版本参数不同',prior_hash['v0001']!=prior_hash['v0002'])
    workload=make_slo_workload('overload',25)
    c.ok('合成流量25条',len(workload)==25)
    c.ok('不同主体都在负载内',{'alice','bob'}=={r.principal for r in workload})
    c.ok('到达tick非降序',all(a.arrival_tick<=b.arrival_tick for a,b in zip(workload,workload[1:])))
    c.ok('所有请求独立ID',len({r.sid for r in workload})==25)
    c.ok('SLO截止不晚于硬TTL',all(r.e2e_slo_ticks<=r.patience_ticks for r in workload))
    c.ok('同批过载确有同tick到达',workload[0].arrival_tick==workload[4].arrival_tick)
    c.rejects('未知负载种类拒绝',lambda:make_slo_workload('other',12))
    c.rejects('不足四个请求拒绝',lambda:make_slo_workload('steady',3))
    cfg=SLOConfig(capacity_blocks=36,cache_limit=18,max_active=2,
                  per_principal_active=1,queue_limit=12,max_ticks=85,policy='edf')
    service=new_environment(tok,models,cfg)
    sch=SLOScheduler(service,cfg,workload)
    sch.pending=[workload[1].sid,workload[0].sid]
    sch.rr_next=0
    chosen=sch._choose()
    c.ok('EDF选择最紧TTFT截止',chosen[0]==workload[1].sid)
    c.ok('EDF返回全部pending候选',set(chosen)==set(sch.pending))
    c.ok('规划只读缓存不加载模型',sch.service.pool.cache_usage()['live_blocks']==0)
    sch.pending=[]
    c.rejects('EDF空队列拒绝',lambda:sch._choose())
    c.ok('新建环境中KV池为空',sch.service.final_flush()['live_blocks']==0)
    config=SLOConfig(capacity_blocks=36,cache_limit=18,max_active=2,
                     per_principal_active=1,queue_limit=12,max_ticks=85,policy='fair_edf')
    service=new_environment(tok,models,config)
    fair=SLOScheduler(service,config,workload)
    fair.pending=[workload[1].sid,workload[0].sid]
    fair.rr_next=fair.rr_tenants.index('bob')
    chosen=fair._choose()
    c.ok('公平EDF先轮到bob',fair.by_sid[chosen[0]].principal=='bob')
    c.ok('fair_edf将主体内deadline排序',chosen[0]==workload[0].sid)
    fair.pending=[]
    c.ok('干净环境能完整释放',fair.service.final_flush()['live_blocks']==0)
    # 边界：最早可在t+1流出第一个token，正好赶上要保留，超过才舍弃。
    t0=SLORequest(sid='edge0',arrival_tick=0,principal='alice',body='Hello world!',
                  patience_ticks=8,ttft_slo_ticks=2,e2e_slo_ticks=5)
    t1=SLORequest(sid='edge1',arrival_tick=0,principal='bob',body='Hello world!',
                  patience_ticks=8,ttft_slo_ticks=2,e2e_slo_ticks=5)
    edgecfg=SLOConfig(capacity_blocks=36,cache_limit=10,max_active=1,
                      per_principal_active=1,queue_limit=5,max_ticks=20,shed_unachievable=True)
    es=new_environment(tok,models,edgecfg)
    edge=SLOScheduler(es,edgecfg,[t0,t1])
    edge.pending=['edge0']
    edge._admit_waiting(1)
    c.ok('恰好达到TTFT预算不提前舍弃',edge.slo_shed_count==0)
    c.ok('等号边界能实际入场',len(edge.admitted)==1)
    edge._finish('edge0','cancelled_active',2)
    edge.pending=['edge1']
    edge._admit_waiting(2)
    c.ok('无法赶上TTFT时舍弃',edge.slo_shed_count==1)
    c.ok('SLO舍弃明确终态',edge.terminal['edge1']['reason']=='slo_unachievable_rejected')
    c.ok('SLO舍弃没有分配物理KV',edge.terminal['edge1']['started_tick'] is None)
    edge.pool.audit()
    c.ok('边界实验最终KV清零',es.final_flush()['live_blocks']==0 and not bool(es.pool.data.count_nonzero()))
    # 独立真实CPU运行，不拿理论伪造模型结果。
    names={name:(cfg,req) for name,cfg,req in scenario_specs()}
    cases={name:run_scenario(tok,models,cfg,req) for name,(cfg,req) in names.items()}
    summary=scenario_summary(cases)
    c.ok('八种固定场景',len(cases)==8 and len(summary['table'])==8)
    c.ok('稳定场景无拥塞',cases['steady_fifo']['summary']['timely_completed']==12)
    c.ok('过载FIFO有SLO失败',cases['overload_fifo']['summary']['timely_completed']<25)
    c.ok('过载FIFO出现队列拒绝',cases['overload_fifo']['summary']['queue_rejections']>0)
    c.ok('RR可跳过同租户受阻的队头',cases['overload_tenant_rr']['summary']['timely_completed']>
          cases['overload_fifo']['summary']['timely_completed'])
    c.ok('舍弃场景实际发生早拒',cases['overload_fair_edf_shed']['summary']['slo_shed_rejections']>0)
    c.ok('舍弃后的全到达比例不冒充100%',cases['overload_fair_edf_shed']['summary']['slo_attainment_all_arrivals']<1)
    c.ok('舍弃后的完成者条件比例可更高',cases['overload_fair_edf_shed']['summary']['slo_attainment_completed_only']>
          cases['overload_fair_edf_shed']['summary']['slo_attainment_all_arrivals'])
    c.ok('相同到达负载的公平EDF与shed相同请求',
          cases['overload_fair_edf']['slo_contract']==cases['overload_fair_edf_shed']['slo_contract'])
    c.ok('提前舍弃可提高满足SLO数但非普遍保证',cases['overload_fair_edf_shed']['summary']['timely_completed']>
          cases['overload_fair_edf']['summary']['timely_completed'])
    c.ok('混合deadline早拒触发',cases['mixed_fair_edf_shed']['summary']['slo_shed_rejections']>0)
    c.ok('混合deadline同请求集合对照',cases['mixed_fair_edf']['slo_contract']==cases['mixed_fair_edf_shed']['slo_contract'])
    c.ok('两组完成分母均不大于到达',all(r['summary']['completed']<=r['summary']['requests'] for r in cases.values()))
    c.ok('完成者成功不大于全部完成',all(r['summary']['timely_completed']<=r['summary']['completed'] for r in cases.values()))
    c.ok('每个请求恰好一个终态',all(len({x['sid'] for x in r['terminal']})==len(r['terminal'])==r['summary']['requests'] for r in cases.values()))
    c.ok('每个场景SLO及时数能从逐请求重算',all(
        sum(slo_passes(row,next(x for x in names[name][1] if x.sid==row['sid']))
            for row in report['terminal'])==report['summary']['timely_completed']
        for name,report in cases.items()))
    c.ok('全部场景真实KV最终清零',all(r['final_flush']['all_bytes_zero'] for r in cases.values()))
    c.ok('没有物理页超配',all(r['summary']['physical_live_blocks_peak']<=
                           r['summary']['physical_capacity_blocks'] for r in cases.values()))
    c.ok('全部场景缓存提示token守恒',all(r['summary']['prefill_tokens_computed']+
          r['summary']['prefill_tokens_reused']==r['summary']['prefill_tokens_total'] for r in cases.values()))
    c.ok('真实增量与完整logits一致',all(r['summary']['max_logit_abs_delta']<2e-5 for r in cases.values()))
    c.ok('原始提示不出现在终态或SLO表',all('body' not in t and 'messages' not in t for r in cases.values() for t in r['terminal']))
    c.ok('队列没有突破硬限制',all(r['summary']['queue_peak']<=r['config']['queue_limit'] for r in cases.values()))
    c.ok('运行中并发没有突破上限',all(r['summary']['active_peak']<=r['config']['max_active'] for r in cases.values()))
    c.ok('完成请求E2E与TTFT的P99遵守nearest rank',all(
        s['latency']['e2e_p99']==nearest_rank(s['latency']['e2e_ticks'],.99) and
        s['latency']['ttft_p99']==nearest_rank(s['latency']['ttft_ticks'],.99)
        for s in (r['summary'] for r in cases.values())))
    c.ok('各租户计数分母合计全部到达',all(sum(v['arrived'] for v in r['summary']['slo_by_tenant'].values())==r['summary']['requests'] for r in cases.values()))
    c.ok('各租户及时成功数合计总体',all(sum(v['slo_good'] for v in r['summary']['slo_by_tenant'].values())==r['summary']['timely_completed'] for r in cases.values()))
    c.ok('Jain公平性只取真实有限比例',all(0<r['summary']['jain_slo_attainment']<=1 for r in cases.values()))
    c.ok('基座和LoRA权重全部未修改',all(state_hash(models[k].state_dict())==prior_hash[k] for k in models))
    c.ok('逻辑tick没有伪装为真实ms',all('milliseconds' not in r['summary'] for r in cases.values()))
    c.ok('早拒记录具有最早输出证明',all(ev['earliest_first_output_tick']>ev['absolute_ttft_deadline']
        for r in cases.values() for ev in r['slo_decisions']))
    c.ok('早拒计数与事件条数一致',all(r['summary']['slo_shed_rejections']==len(r['slo_decisions']) for r in cases.values()))
    c.ok('报告没有真实用户文本',all('Hello world!' not in canonical(r['terminal']) for r in cases.values()))
    c.ok('每场景goodput非负',all(r['summary']['slo_goodput_per_logical_tick']>=0 for r in cases.values()))
    return {'passed':len(c.names),'names':c.names,
            'scope':'真实离线CPU Transformer/KV；抽象逻辑tick调度。非GPU/生产SLO时延成绩'}

In [28]:
checks=run_checks()
print('实际固定检查通过:',checks['passed'],'项')
if run_folder is not None:
    write_json(run_folder/'checks.json',checks)

实际固定检查通过: 91 项


## 17. 独立练习（默认关闭）

**A：所有到达的 SLO 成功率**。实现 `attainment_ex(timely,arrivals)`：空分母 `None`，非法比例明确拒绝。不要把已完成者计作全部到达。

**B：尾分位 nearest-rank**。实现 `percentile_ex(values,p)`：排序后选择第 `ceil(p*n)` 个，空列表 `None`。不要与 NumPy 插值分位数混淆。

**C：TTFT 准入边界**。实现 `can_still_meet_ttft_ex(tick,arrival,budget)`：本课最快的第一 token 发生在准入后下一 tick；恰好等于截止应允许。

三题的测试与参考实现独立命名，不覆盖你的函数。

In [29]:
# 独立练习与参考答案（练习默认关闭）
def exercise_a(fn):
    require(fn(3,8)==.375,'全到达请求是分母')
    require(fn(0,0) is None,'空分母不能伪造0或100%')
    try:fn(9,8)
    except (LessonError,ValueError):pass
    else:raise AssertionError('成功数超过到达数应拒绝')
    return {'exercise':'A','passed':True}


def exercise_b(fn):
    require(fn([1,2,4,8,9],.95)==9,'P95 nearest-rank失败')
    require(fn([1,2,4,8,9],.5)==4,'P50失败')
    require(fn([],.99) is None,'空样本应返回None')
    return {'exercise':'B','passed':True}


def exercise_c(fn):
    require(fn(3,0,2)==False,'t=3下最早首token4，TTFT预算2应拒绝')
    require(fn(1,0,2)==True,'t=1下最早首token2，边界允许')
    require(fn(0,0,2)==True,'刚到达不应拒绝')
    try:fn(-1,0,2)
    except (LessonError,ValueError):pass
    else:raise AssertionError('负tick应拒绝')
    return {'exercise':'C','passed':True}

In [30]:
def attainment_ex(timely_completed, all_arrivals):
    raise NotImplementedError('请实现按全到达为分母的SLO达标率')

RUN_EXERCISE_A=False
if RUN_EXERCISE_A:
    print(exercise_a(attainment_ex))

In [31]:
def percentile_ex(values,p):
    raise NotImplementedError('请实现nearest-rank百分位')

RUN_EXERCISE_B=False
if RUN_EXERCISE_B:
    print(exercise_b(percentile_ex))

In [32]:
def can_still_meet_ttft_ex(current_tick,arrival_tick,ttft_budget):
    raise NotImplementedError('请实现首token最早到达预算检查')

RUN_EXERCISE_C=False
if RUN_EXERCISE_C:
    print(exercise_c(can_still_meet_ttft_ex))

### 完整参考答案

应先完成上面的独立函数再看。参考答案沿用本课契约，测试的是指标计算与逻辑准入，不是模型训练。

In [33]:
def answer_a(timely_completed:int,all_arrivals:int):
    return honest_rate(timely_completed,all_arrivals)


def answer_b(values:list[int],p:float):
    return nearest_rank(values,p)


def answer_c(current_tick:int,arrival_tick:int,ttft_budget:int):
    require(all(type(x) is int for x in (current_tick,arrival_tick,ttft_budget)) and
            0<=arrival_tick<=current_tick and ttft_budget>=1,
            'tick与SLO预算非法')
    return current_tick+1-arrival_tick<=ttft_budget

In [34]:
print('A',exercise_a(answer_a))
print('B',exercise_b(answer_b))
print('C',exercise_c(answer_c))

A {'exercise': 'A', 'passed': True}
B {'exercise': 'B', 'passed': True}
C {'exercise': 'C', 'passed': True}


## 18. CLI 与后续实验

脚本 CLI 实现单独存在，Notebook 也内嵌该函数；在 Notebook 环境**不会自动解析内核参数或重跑训练**。

```bash
python lesson_76_slo_tail_latency_admission_fairness.py --checks-only
python lesson_76_slo_tail_latency_admission_fairness.py --math-only
python lesson_76_slo_tail_latency_admission_fairness.py
python lesson_76_slo_tail_latency_admission_fairness.py --reload-run lesson_76_outputs/实际运行目录
```

本课提供同一模型、同一物理KV、同一到达数据上的对照。但不同策略可能带来不同的 prompt 缓存预热与实际模型计算量，因此不能把本课的完成数量直接推广为**真实 GPU 吞吐或绝对延迟结论**。并且没有做大量独立工作负载种子或真实流量实验。

**进一步实验建议：** 预先设计不同的用户权重、队列上限和 TTFT 预算；把拒绝计数和所有到达请求的成功率始终列在一起。不要通过删除失败请求来制造漂亮的 P99。

下一课：**第77课：过载保护、令牌桶、熔断与重试风暴**。把入口到达频率、幂等标识和拒绝后客户端重试纳入一个可核算系统。

In [35]:
# 命令行入口
def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--checks-only',action='store_true')
    parser.add_argument('--math-only',action='store_true')
    parser.add_argument('--reload-run',type=Path)
    parser.add_argument('--output-dir',type=Path,default=Path('lesson_76_outputs'))
    parser.add_argument('--seed',type=int,default=7601)
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    if args.reload_run:
        print(json.dumps(reload_run(args.reload_run),ensure_ascii=False,indent=2))
        return 0
    if args.math_only:
        print(json.dumps({'q95':nearest_rank([1,3,6,8,9],.95),
                          'q99_small_sample':nearest_rank([1,3,6,8,9],.99),
                          'all_arrivals':honest_rate(3,8),
                          'conditional_served':honest_rate(3,3),
                          'boundary_t1_a0_ttf2':answer_c(1,0,2),
                          'boundary_t2_a0_ttf2':answer_c(2,0,2)},ensure_ascii=False,indent=2))
        return 0
    checks=run_checks()
    print(f"固定检查通过：{checks['passed']}项；CPU真实KV+逻辑tick，不是GPU时延成绩。")
    if args.checks_only:return 0
    folder=fixture_demo(args.output_dir,args.seed)
    write_json(folder/'checks.json',checks)
    return 0

## 主要参考资料

- [vLLM Benchmark：TTFT/TPOT/ITL口径与缓存预热提醒](https://docs.vllm.ai/en/latest/benchmarking/cli/)
- [vLLM Per-Request Metrics](https://docs.vllm.ai/en/latest/features/per_request_metrics/)
- [NVIDIA Triton Batcher：优先级、队列延迟、超时](https://docs.nvidia.com/deeplearning/triton-inference-server/archives/triton-inference-server-2580/user-guide/docs/user_guide/batcher.html)
- [Google SRE：Handling Overload](https://sre.google/sre-book/handling-overload/)
- [PagedAttention](https://arxiv.org/abs/2309.06180)
- [vLLM 前缀缓存设计](https://docs.vllm.ai/en/latest/design/prefix_caching/)
- [PyTorch 随机性与可复现性](https://docs.pytorch.org/docs/stable/notes/randomness.html)

所有模型输出与报告只来自本课代码的本地执行；外部资料用于解释术语、工程约束与边界，文献性能数据不作为本课结果。